In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate PharmaOpt project
cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "fda_outcome_definition"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PHARMAOPT — FDA OUTCOME DEFINITION")
print("=" * 55)
print("Project root:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)
print("Pandas version:", pd.__version__)

In [ ]:
# CELL 2 — LOAD CORRECTED CMS DATASET

STRICT_DIR = PROJECT_ROOT / "outputs" / "cms_strict_review_v01"

SOURCE = STRICT_DIR / "pharmaopt_integrated_package_year_v05.csv"

if not SOURCE.is_file():
    raise FileNotFoundError(
        f"Corrected dataset not found: {SOURCE}\n"
        "Do not substitute the older dataset."
    )

panel = pd.read_csv(
    SOURCE,
    usecols=[
        "package_ndc",
        "prediction_year",
        "shortage_target"
    ],
    dtype={
        "package_ndc": "string",
        "prediction_year": "int16",
        "shortage_target": "int8"
    },
    low_memory=False
)

panel["package_ndc"] = panel["package_ndc"].str.strip()

assert len(panel) == 762907
assert not panel[["package_ndc", "prediction_year"]].duplicated().any()
assert panel["package_ndc"].notna().all()
assert panel["shortage_target"].isin([0, 1]).all()

print("CORRECTED DATASET LOADED")
print("Total observations:", f"{len(panel):,}")

display(
    panel.groupby("prediction_year").agg(
        packages=("package_ndc", "size"),
        original_shortage_labels=("shortage_target", "sum")
    )
)

In [ ]:
# CELL 3 — FDA EVENT EXTRACTION

import mysql.connector
from getpass import getpass

password = getpass("Enter MySQL password: ")

try:
    connection = mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="root",
        password=password,
        database="pharmaopt"
    )
finally:
    del password

try:
    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            linked_package_ndc,
            YEAR(initial_posting_date) AS prediction_year,
            status,
            availability,
            update_type,
            initial_posting_date
        FROM shortage_event
        WHERE linked_package_ndc IS NOT NULL
          AND initial_posting_date >= '2022-01-01'
          AND initial_posting_date < '2026-01-01'
    """)

    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]

    cursor.close()

finally:
    connection.close()

events = pd.DataFrame(rows, columns=columns)

events["linked_package_ndc"] = (
    events["linked_package_ndc"].astype("string").str.strip()
)

events["prediction_year"] = (
    pd.to_numeric(events["prediction_year"]).astype("int16")
)

events["status_normalized"] = (
    events["status"].astype("string").str.strip().str.casefold()
)

print("FDA EVENT DATA LOADED")
print("Linked event records:", f"{len(events):,}")

display(
    events.groupby(
        ["prediction_year", "status"],
        dropna=False
    ).size().rename("records").reset_index()
)

In [ ]:
# CELL 4 — FDA OUTCOME CLASSIFICATION

events["event_any"] = 1

events["event_discontinuation"] = (
    events["status_normalized"]
    .eq("to be discontinued")
    .fillna(False)
    .astype("int8")
)

events["event_non_discontinuation"] = (
    events["status_normalized"]
    .isin(["current", "resolved"])
    .astype("int8")
)

events["event_status_unknown"] = (
    ~events["status_normalized"].isin(
        ["current", "resolved", "to be discontinued"]
    )
).astype("int8")

# Aggregate multiple FDA records for the same package-year.

flags = (
    events.groupby(
        ["linked_package_ndc", "prediction_year"],
        as_index=False
    )
    .agg(
        fda_event_any=("event_any", "max"),
        discontinuation_notice_candidate=(
            "event_discontinuation", "max"
        ),
        non_discontinuation_event_candidate=(
            "event_non_discontinuation", "max"
        ),
        status_unknown_event=(
            "event_status_unknown", "max"
        ),
        fda_event_record_count=("event_any", "size")
    )
)

flags = flags.rename(
    columns={"linked_package_ndc": "package_ndc"}
)

assert not flags[
    ["package_ndc", "prediction_year"]
].duplicated().any()

print("FDA OUTCOME FLAGS CREATED")
print("Unique FDA-linked package-years:", f"{len(flags):,}")

In [ ]:
# CELL 5 — MERGE AND VALIDATE FDA OUTCOMES

result = panel.merge(
    flags,
    how="left",
    on=["package_ndc", "prediction_year"],
    validate="one_to_one"
)

flag_columns = [
    "fda_event_any",
    "discontinuation_notice_candidate",
    "non_discontinuation_event_candidate",
    "status_unknown_event",
    "fda_event_record_count"
]

result[flag_columns] = (
    result[flag_columns].fillna(0).astype("int32")
)

# Verify the original dataset remains intact.

assert len(result) == len(panel)

assert result["shortage_target"].equals(
    panel["shortage_target"]
)

mismatch_count = int(
    (
        result["shortage_target"] != result["fda_event_any"]
    ).sum()
)

print("DATASET INTEGRITY CHECK")
print("=" * 55)
print("Original observations:", f"{len(panel):,}")
print("Updated observations:", f"{len(result):,}")
print("Original target changed: NO")
print("Target recreation mismatches:", mismatch_count)

if mismatch_count != 0:
    print("WARNING: Investigate target mismatches before modeling.")
else:
    print("Original target recreation: PASSED")

In [ ]:
# CELL 6 — FDA OUTCOME SUMMARY

summary = (
    result.groupby("prediction_year", as_index=False)
    .agg(
        package_years=("package_ndc", "size"),
        original_any_event=("shortage_target", "sum"),
        discontinuation_notices=(
            "discontinuation_notice_candidate", "sum"
        ),
        non_discontinuation_candidates=(
            "non_discontinuation_event_candidate", "sum"
        ),
        unknown_status_events=(
            "status_unknown_event", "sum"
        )
    )
)

both_types = (
    (result["discontinuation_notice_candidate"] == 1)
    & (result["non_discontinuation_event_candidate"] == 1)
)

both_counts = (
    result.assign(both_candidate_types=both_types.astype(int))
    .groupby("prediction_year")["both_candidate_types"]
    .sum()
    .reset_index()
)

summary = summary.merge(
    both_counts,
    on="prediction_year",
    validate="one_to_one"
)

print("=" * 65)
print("PHARMAOPT — FDA OUTCOME DEFINITION RESULTS")
print("=" * 65)

display(summary)

print("Total package-year observations:", f"{len(result):,}")
print("Original shortage labels:", int(result["shortage_target"].sum()))
print("Discontinuation candidates:",
      int(result["discontinuation_notice_candidate"].sum()))
print("Non-discontinuation candidates:",
      int(result["non_discontinuation_event_candidate"].sum()))

In [ ]:
# CELL 7 — SAVE FDA OUTCOME DEFINITIONS

if mismatch_count != 0:
    raise ValueError(
        f"Cannot export: {mismatch_count} original target mismatches."
    )

output_file = OUTPUT_DIR / "pharmaopt_fda_outcome_labels_v01.csv"

result.to_csv(output_file, index=False)

print("FDA OUTCOME LABELS SAVED")
print("File:", output_file)
print("Rows:", f"{len(result):,}")
print("Original target preserved:", True)

In [ ]:
# CELL 8 — LOAD EXISTING MODEL SCORES

from sklearn.metrics import average_precision_score, roc_auc_score

SCORES_FILE = (
    PROJECT_ROOT
    / "outputs"
    / "cms_strict_review_v01"
    / "pharmaopt_2025_model_scores_v05.csv"
)

if not SCORES_FILE.exists():
    raise FileNotFoundError(
        f"Strict-CMS score file not found: {SCORES_FILE}"
    )

scores = pd.read_csv(
    SCORES_FILE,
    dtype={"package_ndc": "string"},
    low_memory=False
)

scores["package_ndc"] = (
    scores["package_ndc"].str.strip()
)

required_columns = [
    "package_ndc",
    "prediction_year",
    "shortage_target",
    "risk_score_v04",
    "risk_score_v05"
]

missing = set(required_columns) - set(scores.columns)

if missing:
    raise ValueError(f"Missing score columns: {missing}")

assert len(scores) == 221667
assert scores["prediction_year"].eq(2025).all()
assert not scores["package_ndc"].duplicated().any()
assert scores["shortage_target"].sum() == 96

print("STRICT-CMS MODEL SCORES LOADED")
print("Packages:", f"{len(scores):,}")
print("Original positive labels:", int(scores["shortage_target"].sum()))

In [ ]:
# CELL 9 — MATCH REFINED LABELS TO MODEL SCORES

labels_2025 = result.loc[
    result["prediction_year"] == 2025,
    [
        "package_ndc",
        "shortage_target",
        "discontinuation_notice_candidate",
        "non_discontinuation_event_candidate"
    ]
].copy()

comparison = scores.merge(
    labels_2025,
    on="package_ndc",
    how="left",
    validate="one_to_one",
    suffixes=("_scores", "_labels"),
    indicator=True
)

assert comparison["_merge"].eq("both").all()

assert (
    comparison["shortage_target_scores"]
    == comparison["shortage_target_labels"]
).all()

assert (
    comparison["discontinuation_notice_candidate"]
    + comparison["non_discontinuation_event_candidate"]
    == comparison["shortage_target_labels"]
).all()

assert int(
    comparison["non_discontinuation_event_candidate"].sum()
) == 12

assert int(
    comparison["discontinuation_notice_candidate"].sum()
) == 84

print("REFINED LABEL JOIN PASSED")
print("Packages:", f"{len(comparison):,}")
print("Original positives:", int(comparison["shortage_target_labels"].sum()))
print("Discontinuation candidates:", int(comparison["discontinuation_notice_candidate"].sum()))
print("Non-discontinuation candidates:", int(comparison["non_discontinuation_event_candidate"].sum()))

In [ ]:
# CELL 10 — TARGET-SENSITIVITY EVALUATION

def evaluate_ranking(df, target_col, score_col):
    y = df[target_col].to_numpy(dtype=int)
    s = pd.to_numeric(
        df[score_col], errors="raise"
    ).to_numpy(dtype=float)

    if not np.isfinite(s).all():
        raise ValueError("Invalid model scores detected")

    if len(np.unique(y)) < 2:
        raise ValueError("Both positive and negative labels required")

    package_ids = df["package_ndc"].astype(str).to_numpy()

    # Highest score first; NDC breaks ties deterministically.
    order = np.lexsort((package_ids, -s))

    baseline = y.mean()

    return {
        "target": target_col,
        "model": score_col,
        "positives": int(y.sum()),
        "positive_rate": baseline,
        "average_precision": average_precision_score(y, s),
        "roc_auc": roc_auc_score(y, s),
        "top_500_captured": int(y[order[:500]].sum()),
        "top_1000_captured": int(y[order[:1000]].sum())
    }


results = []

for target in [
    "shortage_target_labels",
    "non_discontinuation_event_candidate"
]:
    for model in [
        "risk_score_v04",
        "risk_score_v05"
    ]:
        results.append(
            evaluate_ranking(comparison, target, model)
        )

evaluation = pd.DataFrame(results)

print("=" * 65)
print("PHARMAOPT — REFINED TARGET MODEL COMPARISON")
print("=" * 65)

display(evaluation.round(6))

In [ ]:
# CELL 11 — SAVE MODEL COMPARISON

comparison_output = (
    OUTPUT_DIR / "pharmaopt_refined_target_model_comparison_v01.csv"
)

evaluation.to_csv(comparison_output, index=False)

print("MODEL COMPARISON SAVED")
print("File:", comparison_output)
print("Models evaluated:", evaluation["model"].nunique())
print("Target definitions:", evaluation["target"].nunique())

In [ ]:
# CELL 12 — REFINED MODEL TRAINING READINESS

train_data = result.loc[
    result["prediction_year"] < 2025
].copy()

test_data = result.loc[
    result["prediction_year"] == 2025
].copy()

targets = [
    "shortage_target",
    "non_discontinuation_event_candidate",
    "discontinuation_notice_candidate"
]

readiness = pd.DataFrame({
    "target": targets,
    "train_positives": [
        int(train_data[c].sum()) for c in targets
    ],
    "test_positives": [
        int(test_data[c].sum()) for c in targets
    ],
    "train_rows": len(train_data),
    "test_rows": len(test_data)
})

print("=" * 60)
print("PHARMAOPT — REFINED MODEL TRAINING READINESS")
print("=" * 60)

display(readiness)

print("\nTraining years: 2022–2024")
print("Evaluation year: 2025")

In [ ]:
# CELL 13 — MODEL FEATURE REVIEW

feature_groups = {
    "Product information": [
        "product_age_days",
        "dosage_form",
        "marketing_category"
    ],
    "Historical recall information": [
        "prior_recall_count",
        "prior_class_I_count",
        "prior_class_II_count",
        "prior_class_III_count",
        "recalls_prior_1y",
        "recalls_prior_2y",
        "recalls_prior_5y",
        "days_since_last_recall"
    ],
    "Historical shortage information": [
        "package_prior_shortages",
        "product_prior_shortages",
        "generic_prior_shortages",
        "labeler_prior_shortages",
        "generic_shortages_prior_1y",
        "labeler_shortages_prior_1y",
        "package_days_since_shortage",
        "generic_days_since_shortage"
    ],
    "CMS demand information": [
        "log_cms_claims",
        "log_cms_fills",
        "log_cms_cost",
        "cms_brand_count",
        "cms_claim_growth",
        "cms_fill_growth",
        "cms_demand_match"
    ],
    "Additional v0.5 features": [
        "linked_recall_history_record_3y",
        "prior_recall_count_3y"
    ]
}

feature_review = pd.DataFrame([
    {
        "group": group,
        "feature": feature
    }
    for group, features in feature_groups.items()
    for feature in features
])

print("PHARMAOPT — MODEL FEATURE REVIEW")
print("=" * 60)

display(feature_review)

print("Features listed:", len(feature_review))

In [ ]:
# CELL 14 — PHARMAOPT RISK PRIORITY LIST

from pathlib import Path

priority = comparison.copy()

# Select the existing v0.4 model as our baseline
priority["risk_score"] = pd.to_numeric(
    priority["risk_score_v04"],
    errors="raise"
)

# Rank packages from highest to lowest risk
priority = priority.sort_values(
    ["risk_score", "package_ndc"],
    ascending=[False, True]
).reset_index(drop=True)

priority["risk_rank"] = np.arange(1, len(priority) + 1)

# Create a top-500 priority list
top_500 = priority.head(500).copy()

# Summarize how the ranking performed
summary = pd.DataFrame({
    "Metric": [
        "Total packages evaluated",
        "Packages prioritized",
        "Original FDA events captured",
        "Non-discontinuation candidates captured",
        "Discontinuation candidates captured",
        "Percentage of packages prioritized"
    ],
    "Value": [
        len(priority),
        len(top_500),
        int(top_500["shortage_target_labels"].sum()),
        int(top_500["non_discontinuation_event_candidate"].sum()),
        int(top_500["discontinuation_notice_candidate"].sum()),
        round(100 * len(top_500) / len(priority), 4)
    ]
})

# Save a reproducible prototype output
priority_columns = [
    "package_ndc",
    "prediction_year",
    "risk_score",
    "risk_rank",
    "shortage_target_labels",
    "non_discontinuation_event_candidate",
    "discontinuation_notice_candidate"
]

output_path = OUTPUT_DIR / "pharmaopt_2025_top500_priority_v01.csv"

top_500[priority_columns].to_csv(
    output_path,
    index=False
)

print("=" * 60)
print("PHARMAOPT — 2025 RISK PRIORITY PROTOTYPE")
print("=" * 60)

display(summary)

print("\nTOP 10 HIGHEST-RANKED PACKAGES")
display(top_500[priority_columns].head(10))

print("\nSaved to:", output_path)

In [ ]:
# CELL 15 — RESOURCE ALLOCATION SIMULATION

budgets = [100, 250, 500, 1000, 2500]

targets = {
    "Original FDA events": "shortage_target_labels",
    "Non-discontinuation events": "non_discontinuation_event_candidate",
    "Discontinuation notices": "discontinuation_notice_candidate"
}

allocation_rows = []

for model in ["risk_score_v04", "risk_score_v05"]:

    ranked = comparison.sort_values(
        [model, "package_ndc"],
        ascending=[False, True]
    ).reset_index(drop=True)

    for budget in budgets:
        selected = ranked.head(budget)

        for target_name, target_col in targets.items():

            total_events = int(comparison[target_col].sum())
            captured = int(selected[target_col].sum())

            random_expected = (
                budget * total_events / len(comparison)
            )

            allocation_rows.append({
                "model": model,
                "review_capacity": budget,
                "target": target_name,
                "events_captured": captured,
                "total_events": total_events,
                "recall": captured / total_events if total_events else np.nan,
                "precision": captured / budget,
                "random_expected_captures": random_expected,
                "lift_vs_random": (
                    captured / random_expected
                    if random_expected > 0 else np.nan
                )
            })

allocation_results = pd.DataFrame(allocation_rows)

print("=" * 65)
print("PHARMAOPT — RESOURCE ALLOCATION SIMULATION")
print("=" * 65)

display(
    allocation_results.loc[
        allocation_results["target"] == "Non-discontinuation events"
    ].round(4)
)

In [ ]:
# CELL 16 — CREATE OPERATIONAL REVIEW PLAN

REVIEW_CAPACITY = 500
HYPOTHETICAL_COST_PER_REVIEW = 1000

allocation_plan = priority.head(REVIEW_CAPACITY).copy()

allocation_plan["review_cost_assumed_usd"] = (
    HYPOTHETICAL_COST_PER_REVIEW
)

allocation_plan["cumulative_review_cost_usd"] = (
    allocation_plan["review_cost_assumed_usd"].cumsum()
)

allocation_plan["review_status"] = "Not reviewed"

allocation_plan["priority_tier"] = pd.cut(
    allocation_plan["risk_rank"],
    bins=[0, 100, 250, 500],
    labels=["Tier 1", "Tier 2", "Tier 3"]
)

plan_columns = [
    "package_ndc",
    "prediction_year",
    "risk_rank",
    "risk_score",
    "priority_tier",
    "review_cost_assumed_usd",
    "cumulative_review_cost_usd",
    "review_status"
]

allocation_file = (
    OUTPUT_DIR / "pharmaopt_2025_allocation_plan_v01.csv"
)

allocation_plan[plan_columns].to_csv(
    allocation_file,
    index=False
)

allocation_results.to_csv(
    OUTPUT_DIR / "pharmaopt_allocation_simulation_v01.csv",
    index=False
)

print("=" * 60)
print("PHARMAOPT — ALLOCATION PLAN CREATED")
print("=" * 60)

print("Review capacity:", REVIEW_CAPACITY)
print(
    "Hypothetical budget: $",
    f"{REVIEW_CAPACITY * HYPOTHETICAL_COST_PER_REVIEW:,}",
    sep=""
)

print("Packages assigned:", len(allocation_plan))
print("Saved:", allocation_file)

display(allocation_plan[plan_columns].head(10))

In [ ]:
# CELL 17 — EXPORT FULL MODEL RANKINGS FOR DASHBOARD

dashboard_data = comparison[
    [
        "package_ndc",
        "prediction_year",
        "risk_score_v04",
        "risk_score_v05",
        "shortage_target_labels",
        "non_discontinuation_event_candidate",
        "discontinuation_notice_candidate"
    ]
].copy()

dashboard_data = dashboard_data.rename(
    columns={"shortage_target_labels": "fda_event_any"}
)

dashboard_data["package_ndc"] = (
    dashboard_data["package_ndc"].astype("string").str.strip()
)

assert len(dashboard_data) == 221667
assert not dashboard_data["package_ndc"].duplicated().any()

dashboard_file = (
    OUTPUT_DIR / "pharmaopt_2025_dashboard_full_v01.csv"
)

dashboard_data.to_csv(dashboard_file, index=False)

print("=" * 60)
print("PHARMAOPT — DASHBOARD DATA EXPORT COMPLETE")
print("=" * 60)
print("Packages exported:", f"{len(dashboard_data):,}")
print("Model versions:", "v0.4 and v0.5")
print("Saved:", dashboard_file)

In [ ]:
# CELL 18 — FDA PRODUCT DIRECTORY STRUCTURE

import mysql.connector
from getpass import getpass

password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="pharmaopt"
)

del password

cursor = connection.cursor()

cursor.execute("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'pharmaopt'
      AND table_name IN ('product', 'package')
    ORDER BY table_name, ordinal_position
""")

schema = pd.DataFrame(
    cursor.fetchall(),
    columns=["table_name", "column_name", "data_type"]
)

display(schema)

print("PRODUCT DIRECTORY STRUCTURE LOADED")

In [ ]:
# CELL 19 — IDENTIFY DRUG METADATA FIELDS

columns_by_table = {
    table: schema.loc[
        schema["table_name"] == table,
        "column_name"
    ].tolist()
    for table in ["product", "package"]
}

for table, columns in columns_by_table.items():
    print(f"\n{table.upper()} TABLE")
    print("-" * 45)
    for column in columns:
        print(column)

print("\nNext: Join FDA package NDCs to product metadata.")

In [ ]:
# CELL 20 — DISCOVER ACTUAL MYSQL TABLES

cursor.execute("SELECT DATABASE()")
print("Connected database:", cursor.fetchone()[0])

cursor.execute("SHOW FULL TABLES")

tables = cursor.fetchall()

print("\nTABLES FOUND:")
print("=" * 50)

for table in tables:
    print(table[0])

print("\nTotal tables:", len(tables))

In [ ]:
# CELL 21 — INSPECT DRUG PRODUCT AND PACKAGE TABLES

cursor.execute("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'pharmaopt'
      AND table_name IN ('drug_product', 'drug_package')
    ORDER BY table_name, ordinal_position
""")

drug_schema = pd.DataFrame(
    cursor.fetchall(),
    columns=["table_name", "column_name", "data_type"]
)

print("=" * 55)
print("PHARMAOPT — DRUG DIRECTORY SCHEMA")
print("=" * 55)

for table in ["drug_product", "drug_package"]:
    print(f"\n{table.upper()}")
    print("-" * 40)

    subset = drug_schema[
        drug_schema["table_name"] == table
    ]

    for _, row in subset.iterrows():
        print(f"{row['column_name']} ({row['data_type']})")

In [ ]:
# CELL 22 — LOAD FDA DRUG DIRECTORY

drug_query = """
SELECT
    pkg.package_ndc,
    pkg.product_id,
    pkg.package_description,
    prod.generic_name,
    prod.brand_name,
    prod.labeler_name,
    prod.dosage_form,
    prod.marketing_category
FROM drug_package AS pkg
LEFT JOIN drug_product AS prod
    ON pkg.product_id = prod.product_id
"""

drug_directory = pd.read_sql(
    drug_query,
    connection
)

drug_directory["package_ndc"] = (
    drug_directory["package_ndc"]
    .astype("string")
    .str.strip()
)

print("=" * 55)
print("PHARMAOPT — FDA DRUG DIRECTORY LOADED")
print("=" * 55)

print("Directory rows:", f"{len(drug_directory):,}")
print(
    "Unique package NDCs:",
    f"{drug_directory['package_ndc'].nunique():,}"
)

display(drug_directory.head(10))

In [ ]:
# CELL 23 — ENRICH MODEL RANKINGS

metadata_columns = [
    "generic_name",
    "brand_name",
    "labeler_name",
    "dosage_form",
    "marketing_category",
    "package_description"
]

directory_clean = drug_directory.copy()

directory_clean = directory_clean.dropna(
    subset=["package_ndc"]
)

directory_clean = directory_clean.sort_values(
    ["package_ndc", "product_id"],
    na_position="last"
)

# Select a consistent record for each package NDC.
# Multiple distinct matches will be reported separately.
duplicate_ndcs = (
    directory_clean["package_ndc"]
    .value_counts()
)

print(
    "Package NDCs with multiple directory records:",
    int((duplicate_ndcs > 1).sum())
)

directory_unique = directory_clean.drop_duplicates(
    subset=["package_ndc"],
    keep="first"
)

enriched = dashboard_data.merge(
    directory_unique[
        ["package_ndc"] + metadata_columns
    ],
    on="package_ndc",
    how="left",
    validate="one_to_one"
)

assert len(enriched) == len(dashboard_data)

for col in metadata_columns:
    enriched[col] = (
        enriched[col]
        .astype("string")
        .str.strip()
        .fillna("Not available")
    )

print("=" * 55)
print("PHARMAOPT — DRUG ENRICHMENT COMPLETE")
print("=" * 55)

print("Total model packages:", f"{len(enriched):,}")

matched = enriched["generic_name"].ne(
    "Not available"
).sum()

print("Packages with generic names:", f"{matched:,}")

print(
    "Generic name coverage:",
    f"{matched / len(enriched):.2%}"
)

display(
    enriched[
        [
            "package_ndc",
            "generic_name",
            "brand_name",
            "labeler_name",
            "dosage_form"
        ]
    ].head(10)
)

In [ ]:
# CELL 24 — EXPORT ENRICHED DASHBOARD DATA

enriched_file = (
    OUTPUT_DIR /
    "pharmaopt_2025_dashboard_enriched_v01.csv"
)

enriched.to_csv(
    enriched_file,
    index=False
)

print("=" * 60)
print("PHARMAOPT — ENRICHED DASHBOARD EXPORT")
print("=" * 60)

print("Rows exported:", f"{len(enriched):,}")
print("Saved to:", enriched_file)

print("\nDRUG INFORMATION COVERAGE")

for col in [
    "generic_name",
    "brand_name",
    "labeler_name",
    "dosage_form"
]:
    available = int(
        enriched[col].ne("Not available").sum()
    )

    print(
        f"{col}: {available:,} "
        f"({available / len(enriched):.1%})"
    )

In [ ]:
import sys
print(sys.executable)

In [ ]:
# CELL 25 — HISTORICAL FEATURE AVAILABILITY AUDIT

import pandas as pd
from pathlib import Path

SOURCE = (
    Path(r"C:\Users\jpiza\PharmaOpt")
    / "outputs"
    / "cms_strict_review_v01"
    / "pharmaopt_integrated_package_year_v05.csv"
)

columns = pd.read_csv(SOURCE, nrows=0).columns.tolist()

print("PHARMAOPT — HISTORICAL FEATURE AUDIT")
print("=" * 55)
print("Total columns:", len(columns))

for index, column in enumerate(columns, start=1):
    print(f"{index:3}. {column}")

In [ ]:
# CELL 26 — TEMPORAL INTEGRITY AUDIT

import pandas as pd
import numpy as np

date_columns = [
    "marketing_start_date",
    "most_recent_prior_recall",
    "package_last_shortage",
    "product_last_shortage",
    "generic_last_shortage",
    "labeler_last_shortage",
]

columns_to_load = [
    "prediction_year",
    "cms_data_year",
    "cms_demand_match",
] + date_columns

audit = pd.read_csv(
    SOURCE,
    usecols=columns_to_load,
    low_memory=False
)

# Prediction cutoff: January 1 of each prediction year.
cutoff = pd.to_datetime(
    audit["prediction_year"].astype(str) + "-01-01"
)

results = []

for column in date_columns:
    dates = pd.to_datetime(
        audit[column],
        errors="coerce"
    )

    available = dates.notna()

    future_dates = (
        available & (dates >= cutoff)
    )

    results.append({
        "feature": column,
        "non_missing_dates": int(available.sum()),
        "dates_on_or_after_cutoff": int(future_dates.sum())
    })

report = pd.DataFrame(results)

print("PHARMAOPT — TEMPORAL INTEGRITY RESULTS")
print("=" * 60)

display(report)

# CMS timing check
cms_year = pd.to_numeric(
    audit["cms_data_year"],
    errors="coerce"
)

matched = pd.to_numeric(
    audit["cms_demand_match"],
    errors="coerce"
).eq(1)

incorrect_cms_lag = (
    matched
    & ((audit["prediction_year"] - cms_year) != 3)
)

print("\nCMS matched rows:", int(matched.sum()))
print("Incorrect 3-year CMS mappings:", int(incorrect_cms_lag.sum()))

print("\nAudit complete. No files modified.")

In [ ]:
# CELL 27 — MARKETING DATE INVESTIGATION

marketing_dates = pd.to_datetime(
    audit["marketing_start_date"],
    errors="coerce"
)

cutoff = pd.to_datetime(
    audit["prediction_year"].astype(str) + "-01-01"
)

future_marketing = audit.loc[
    marketing_dates >= cutoff
].copy()

future_marketing["marketing_start_parsed"] = (
    marketing_dates.loc[future_marketing.index]
)

future_marketing["days_after_cutoff"] = (
    future_marketing["marketing_start_parsed"]
    - cutoff.loc[future_marketing.index]
).dt.days

print("PHARMAOPT — MARKETING DATE INVESTIGATION")
print("=" * 60)

print("Affected package-year rows:", len(future_marketing))

print("\nAffected rows by prediction year:")
display(
    future_marketing.groupby("prediction_year")
    .size()
    .rename("affected_rows")
    .to_frame()
)

print("\nDays after prediction cutoff:")
display(future_marketing["days_after_cutoff"].describe())

print("\nExample affected records:")
display(
    future_marketing[
        [
            "prediction_year",
            "marketing_start_date",
            "days_after_cutoff"
        ]
    ].head(10)
)

print("\nRead-only investigation complete.")

In [ ]:
# CELL 28 — MARKETING DATE AND PRODUCT AGE VALIDATION

import pandas as pd
import numpy as np

df = pd.read_csv(
    SOURCE,
    usecols=[
        "package_ndc",
        "prediction_year",
        "marketing_start_date",
        "product_age_days",
        "shortage_target"
    ],
    dtype={"package_ndc": "string"},
    low_memory=False
)

dates = pd.to_datetime(
    df["marketing_start_date"],
    errors="coerce"
)

cutoff = pd.to_datetime(
    df["prediction_year"].astype(str) + "-01-01"
)

age = pd.to_numeric(
    df["product_age_days"],
    errors="coerce"
)

equal_cutoff = dates.eq(cutoff)
after_cutoff = dates.gt(cutoff)

print("PHARMAOPT — MARKETING DATE VALIDATION")
print("=" * 60)

print("Total observations:", len(df))
print("Dates exactly at cutoff:", int(equal_cutoff.sum()))
print("Dates strictly after cutoff:", int(after_cutoff.sum()))
print("Missing marketing dates:", int(dates.isna().sum()))

print("\nPRODUCT AGE FOR JANUARY 1 RECORDS")
display(
    age.loc[equal_cutoff].describe()
)

print("\nAGE VALUES AT CUTOFF")
display(
    age.loc[equal_cutoff]
    .value_counts(dropna=False)
    .head(10)
)

expected_age = (cutoff - dates).dt.days

age_mismatch = (
    age.notna()
    & expected_age.notna()
    & age.ne(expected_age)
)

print("\nPRODUCT AGE CONSISTENCY")
print("Age/date mismatches:", int(age_mismatch.sum()))
print("Negative product ages:", int(age.lt(0).sum()))

print("\nORIGINAL OUTCOMES AT CUTOFF")
print(
    "Positive FDA-linked flags:",
    int(df.loc[equal_cutoff, "shortage_target"].sum())
)

print("\nRead-only audit complete.")

In [ ]:
# CELL 29 — PRODUCT AGE FORMULA INSPECTION

from pathlib import Path
import json

root = Path(r"C:\Users\jpiza\PharmaOpt")

search_terms = [
    "product_age_days",
    "marketing_start_date"
]

matches = []

for notebook in sorted(root.rglob("*.ipynb")):
    if ".venv" in notebook.parts:
        continue

    try:
        data = json.loads(notebook.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        continue

    for cell_number, cell in enumerate(data.get("cells", []), 1):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(term in source for term in search_terms):
            relevant_lines = [
                line.strip()
                for line in source.splitlines()
                if any(term in line for term in search_terms)
            ]

            matches.append({
                "notebook": notebook.name,
                "cell": cell_number,
                "matching_lines": " | ".join(relevant_lines[:5])
            })

print("PHARMAOPT — PRODUCT AGE FORMULA SEARCH")
print("=" * 60)

for match in matches[:30]:
    print(f"\n{match['notebook']} — Cell {match['cell']}")
    print(match["matching_lines"])

print(f"\nTotal matching cells: {len(matches)}")
print("Read-only inspection complete.")

In [ ]:
# CELL 30 — INSPECT HISTORICAL POPULATION LOGIC

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

# Search project folders for the exact notebook.
matches = [
    p for p in root.rglob(
        "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
    )
    if ".ipynb_checkpoints" not in p.parts
    and ".venv" not in p.parts
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one matching notebook, found {len(matches)}: {matches}"
    )

notebook_path = matches[0]

with open(notebook_path, "r", encoding="utf-8") as file:
    notebook = json.load(file)

cell = notebook["cells"][13]

source_code = "".join(cell["source"])

print("PHARMAOPT — HISTORICAL POPULATION LOGIC")
print("=" * 65)
print("Notebook:", notebook_path)
print()
print(source_code)
print("\nRead-only inspection complete.")

In [ ]:
# CELL 31 — RECALL DATE FIELD INSPECTION

import mysql.connector
from getpass import getpass
import pandas as pd

password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="pharmaopt"
)

del password

try:
    query = """
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'pharmaopt'
          AND table_name = 'recall_event'
        ORDER BY ordinal_position
    """

    cursor = connection.cursor()
    cursor.execute(query)

    columns = pd.DataFrame(
        cursor.fetchall(),
        columns=["column_name", "data_type"]
    )

    cursor.close()

finally:
    connection.close()

print("PHARMAOPT — RECALL DATE FIELD INSPECTION")
print("=" * 60)

display(columns)

print("\nRead-only inspection complete.")

In [ ]:
# CELL 32 — RECALL TIMING LEAKAGE AUDIT

import mysql.connector
from getpass import getpass
import pandas as pd

password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="pharmaopt"
)

del password

try:
    query = """
        SELECT
            re.recall_number,
            re.recall_initiation_date,
            re.report_date,
            rp.linked_package_ndc AS package_ndc
        FROM recall_event re
        INNER JOIN recall_package rp
            ON re.recall_number = rp.recall_number
        WHERE rp.linked_package_ndc IS NOT NULL
    """

    recalls = pd.read_sql(query, connection)

finally:
    connection.close()

recalls["recall_initiation_date"] = pd.to_datetime(
    recalls["recall_initiation_date"],
    errors="coerce"
)

recalls["report_date"] = pd.to_datetime(
    recalls["report_date"],
    errors="coerce"
)

results = []

for year in [2022, 2023, 2024, 2025]:

    cutoff = pd.Timestamp(f"{year}-01-01")

    initiated_before = (
        recalls["recall_initiation_date"] < cutoff
    )

    reported_after = (
        recalls["report_date"] >= cutoff
    )

    missing_report = recalls["report_date"].isna()

    results.append({
        "prediction_year": year,
        "prior_recall_relationships": int(
            initiated_before.sum()
        ),
        "reported_on_or_after_cutoff": int(
            (initiated_before & reported_after).sum()
        ),
        "missing_report_date": int(
            (initiated_before & missing_report).sum()
        ),
        "affected_packages": int(
            recalls.loc[
                initiated_before & reported_after,
                "package_ndc"
            ].nunique()
        )
    })

audit_results = pd.DataFrame(results)

print("PHARMAOPT — RECALL TIMING LEAKAGE AUDIT")
print("=" * 65)

display(audit_results)

print("\nRead-only audit complete.")

In [ ]:
# CELL 33 — RECALL LEAKAGE POPULATION IMPACT

import pandas as pd

panel_keys = pd.read_csv(
    SOURCE,
    usecols=[
        "package_ndc",
        "prediction_year",
        "shortage_target"
    ],
    dtype={"package_ndc": "string"},
    low_memory=False
)

panel_keys["package_ndc"] = (
    panel_keys["package_ndc"].str.strip()
)

recalls["package_ndc"] = (
    recalls["package_ndc"]
    .astype("string")
    .str.strip()
)

results = []

for year in [2022, 2023, 2024, 2025]:

    cutoff = pd.Timestamp(f"{year}-01-01")

    late = recalls.loc[
        (recalls["recall_initiation_date"] < cutoff)
        &
        (recalls["report_date"] >= cutoff)
    ].copy()

    affected_ndcs = set(
        late["package_ndc"].dropna()
    )

    population = panel_keys.loc[
        panel_keys["prediction_year"] == year
    ].copy()

    impacted = population[
        population["package_ndc"].isin(affected_ndcs)
    ]

    results.append({
        "year": year,
        "model_population": len(population),
        "affected_model_packages": len(impacted),
        "affected_percentage": (
            100 * len(impacted) / len(population)
        ),
        "positive_labels_affected": int(
            impacted["shortage_target"].sum()
        ),
        "unique_late_recall_events": (
            late["recall_number"].nunique()
        )
    })

impact = pd.DataFrame(results)

print("PHARMAOPT — RECALL LEAKAGE IMPACT")
print("=" * 65)

display(impact.round(4))

print("\nRead-only validation complete.")

In [ ]:
# CELL 34 — RECALL FEATURE TIMING COMPARISON
# Read-only: no model training or data modification.

import pandas as pd

current_features = pd.read_csv(
    SOURCE,
    usecols=[
        "package_ndc",
        "prediction_year",
        "prior_recall_count",
        "shortage_target"
    ],
    dtype={"package_ndc": "string"},
    low_memory=False
)

current_features["package_ndc"] = (
    current_features["package_ndc"].str.strip()
)

rows = []

for year in [2022, 2023, 2024, 2025]:

    cutoff = pd.Timestamp(f"{year}-01-01")

    historical = recalls.loc[
        (recalls["recall_initiation_date"] < cutoff)
        & (recalls["report_date"] < cutoff)
    ].copy()

    corrected_counts = (
        historical.groupby("package_ndc")["recall_number"]
        .nunique()
        .rename("report_date_safe_count")
        .reset_index()
    )

    population = current_features.loc[
        current_features["prediction_year"] == year
    ].copy()

    check = population.merge(
        corrected_counts,
        on="package_ndc",
        how="left",
        validate="one_to_one"
    )

    check["report_date_safe_count"] = (
        check["report_date_safe_count"].fillna(0).astype(int)
    )

    check["prior_recall_count"] = pd.to_numeric(
        check["prior_recall_count"],
        errors="raise"
    )

    difference = (
        check["prior_recall_count"]
        - check["report_date_safe_count"]
    )

    rows.append({
        "year": year,
        "packages_with_changed_counts": int(difference.ne(0).sum()),
        "counts_decreasing": int(difference.gt(0).sum()),
        "counts_increasing": int(difference.lt(0).sum()),
        "total_count_reduction": int(difference.sum()),
        "largest_count_reduction": int(difference.max())
    })

feature_impact = pd.DataFrame(rows)

print("PHARMAOPT — RECALL FEATURE TIMING COMPARISON")
print("=" * 65)

display(feature_impact)

print("\nRead-only comparison complete.")

In [ ]:
# CELL 35 — FINAL RECALL AUDIT CHECK

assert feature_impact["packages_with_changed_counts"].sum() == 735
assert feature_impact["total_count_reduction"].sum() == 898
assert feature_impact["counts_increasing"].sum() == 0

print("PHARMAOPT — RECALL TIMING AUDIT VERIFIED")
print("Affected package-year rows: 735")
print("Recall-count reduction: 898")
print("Original datasets and models: unchanged")

In [ ]:
# CELL 36 — RECALL FEATURE DEPENDENCY INVENTORY

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

targets = [
    "recall_initiation_date",
    "report_date",
    "prior_class_I_count",
    "prior_recall_count_3y",
    "prior_sterility_risk_count_3y",
]

notebooks = [
    "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb",
    "10_pharmaopt_qualitative_risk_integration.ipynb",
]

for name in notebooks:
    paths = [
        p for p in root.rglob(name)
        if ".ipynb_checkpoints" not in p.parts
        and ".venv" not in p.parts
    ]

    if not paths:
        print(f"\nNOT FOUND: {name}")
        continue

    with open(paths[0], encoding="utf-8") as f:
        data = json.load(f)

    print(f"\nNOTEBOOK: {paths[0].name}")
    print("=" * 65)

    for i, cell in enumerate(data["cells"], start=1):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])
        matches = [
            line.strip()
            for line in source.splitlines()
            if any(term in line for term in targets)
        ]

        if matches:
            print(f"\nCell {i}")
            for line in matches[:12]:
                print(" ", line)

print("\nRead-only dependency inventory complete.")

In [ ]:
# CELL 37 — QUALITATIVE FEATURE TIMING INSPECTION

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

print("PHARMAOPT — QUALITATIVE FEATURE TIMING AUDIT")
print("=" * 65)

print("\nAVAILABLE NOTEBOOK 10 FILES")

for path in sorted(root.rglob("10*.ipynb")):
    if ".venv" not in path.parts:
        if ".ipynb_checkpoints" not in path.parts:
            print(path.relative_to(root))

print("\nNOTEBOOK 11 — CELLS 35–38")

matches = [
    p for p in root.rglob(
        "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
    )
    if ".venv" not in p.parts
    and ".ipynb_checkpoints" not in p.parts
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one Notebook 11, found {len(matches)}"
    )

with open(matches[0], encoding="utf-8") as f:
    notebook = json.load(f)

for cell_number in [35, 36, 37, 38]:

    cell = notebook["cells"][cell_number - 1]

    if cell["cell_type"] != "code":
        continue

    print(f"\n{'=' * 65}")
    print(f"CELL {cell_number}")
    print("=" * 65)

    source = "".join(cell["source"])
    print(source)

print("\nRead-only inspection complete.")

In [ ]:
# CELL 38 — QUALITATIVE FEATURE DEPENDENCY AUDIT

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

matches = [
    p for p in root.rglob(
        "10_qualitative_risk_model_integration.ipynb"
    )
    if ".venv" not in p.parts
    and ".ipynb_checkpoints" not in p.parts
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one Notebook 10, found {len(matches)}"
    )

notebook_path = matches[0]

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

search_terms = [
    "report_date",
    "recall_initiation_date",
    "prediction_year",
    "index_date",
    "cutoff",
    "prior_recall_count_3y",
    "DateOffset",
]

print("PHARMAOPT — QUALITATIVE FEATURE DEPENDENCIES")
print("=" * 65)
print("Notebook:", notebook_path)

matching_cells = 0

for number, cell in enumerate(notebook["cells"], start=1):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    lines = [
        (i, line.strip())
        for i, line in enumerate(source.splitlines(), start=1)
        if any(term in line for term in search_terms)
    ]

    if lines:
        matching_cells += 1

        print(f"\nCELL {number}")
        print("-" * 45)

        for line_number, line in lines[:25]:
            print(f"Line {line_number}: {line}")

print("\nMatching cells:", matching_cells)
print("Read-only audit complete.")

In [ ]:
# CELL 39 — ORIGINAL RECALL FEATURE INVENTORY
# Read-only: inspect existing SQL definitions.

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

matches = [
    p for p in root.rglob(
        "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
    )
    if ".venv" not in p.parts
    and ".ipynb_checkpoints" not in p.parts
]

assert len(matches) == 1

with open(matches[0], encoding="utf-8") as f:
    notebook = json.load(f)

cell = notebook["cells"][13]
source = "".join(cell["source"])

start = source.find("recall_query =")
end = source.find("# 4. MERGE RECALL FEATURES")

if start == -1 or end == -1 or end <= start:
    raise ValueError("Could not locate recall feature logic")

print("PHARMAOPT — ORIGINAL RECALL FEATURE DEFINITIONS")
print("=" * 65)

print(source[start:end])

print("\nRead-only inspection complete.")

In [ ]:
# CELL 40 — RECALL REBUILD PRECHECK
# Read-only. Does not modify any files or models.

import pandas as pd

required = [
    "recall_number",
    "package_ndc",
    "recall_initiation_date",
    "report_date"
]

missing = sorted(set(required) - set(recalls.columns))

if missing:
    raise ValueError(f"Missing recall columns: {missing}")

check_rows = []

for year in [2022, 2023, 2024, 2025]:

    cutoff = pd.Timestamp(f"{year}-01-01")

    original = recalls.loc[
        recalls["recall_initiation_date"] < cutoff
    ]

    corrected = original.loc[
        original["report_date"] < cutoff
    ]

    excluded = original.loc[
        original["report_date"].isna()
        | (original["report_date"] >= cutoff)
    ]

    check_rows.append({
        "year": year,
        "original_relationships": len(original),
        "corrected_relationships": len(corrected),
        "excluded_relationships": len(excluded),
        "excluded_recall_events": excluded["recall_number"].nunique(),
        "excluded_packages": excluded["package_ndc"].nunique()
    })

rebuild_precheck = pd.DataFrame(check_rows)

assert (
    rebuild_precheck["original_relationships"]
    == rebuild_precheck["corrected_relationships"]
    + rebuild_precheck["excluded_relationships"]
).all()

print("PHARMAOPT — RECALL REBUILD PRECHECK")
print("=" * 65)

display(rebuild_precheck)

print("\nValidation passed: original relationships reconcile.")
print("Original datasets and models unchanged.")

In [ ]:
# CELL 41 — CORRECTED RECALL FEATURE CONSTRUCTION
# Read-only source data. No exports or model training.

import pandas as pd
import numpy as np

r = recalls.copy()

r["package_ndc"] = r["package_ndc"].astype("string").str.strip()

# The Cell 32 dataset must include classification for this rebuild.
if "classification" not in r.columns:
    print("Classification field not loaded.")
    print("We need to retrieve it before calculating class-specific features.")
else:
    print("Classification field available.")

print("\nPHARMAOPT — RECALL REBUILD PREPARATION")
print("=" * 60)

print("Recall relationships:", f"{len(r):,}")
print("Unique recall events:", r["recall_number"].nunique())
print("Unique package NDCs:", r["package_ndc"].nunique())

print("\nAvailable fields:")
print(r.columns.tolist())

print("\nNo files or models modified.")

In [ ]:
# CELL 42 — RETRIEVE FDA RECALL CLASSIFICATIONS
# Read-only. Does not modify MySQL or existing datasets.

import pandas as pd
import mysql.connector
from getpass import getpass

password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="pharmaopt"
)

del password

try:
    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            recall_number,
            classification
        FROM recall_event
    """)

    classification_data = pd.DataFrame(
        cursor.fetchall(),
        columns=["recall_number", "classification"]
    )

    cursor.close()

finally:
    connection.close()

# Normalize recall identifiers.
classification_data["recall_number"] = (
    classification_data["recall_number"]
    .astype("string")
    .str.strip()
)

r["recall_number"] = (
    r["recall_number"]
    .astype("string")
    .str.strip()
)

# Verify that each recall has one classification record.
assert not classification_data["recall_number"].duplicated().any()

# Add classification without changing the original recalls table.
r = r.merge(
    classification_data,
    on="recall_number",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("PHARMAOPT — RECALL CLASSIFICATION INTEGRATION")
print("=" * 60)

print("Original relationships:", f"{len(recalls):,}")
print("Updated relationships:", f"{len(r):,}")

print(
    "Matched relationships:",
    int(r["_merge"].eq("both").sum())
)

print(
    "Unmatched relationships:",
    int(r["_merge"].eq("left_only").sum())
)

print("\nRECALL CLASSIFICATION DISTRIBUTION")

display(
    r["classification"]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="relationships")
)

assert len(r) == len(recalls)
assert r["_merge"].eq("both").all()

r = r.drop(columns="_merge")

print("\nClassification integration: PASSED")
print("Original datasets and models: unchanged")

In [ ]:
# CELL 43 — CORRECTED HISTORICAL RECALL FEATURES
# In-memory rebuild; original files remain unchanged.

import pandas as pd
import numpy as np

required = [
    "package_ndc", "recall_number", "classification",
    "recall_initiation_date", "report_date"
]

missing = sorted(set(required) - set(r.columns))
if missing:
    raise ValueError(f"Missing fields: {missing}")

history = r[required].copy()

for col in ["recall_initiation_date", "report_date"]:
    history[col] = pd.to_datetime(history[col], errors="coerce")

history["package_ndc"] = (
    history["package_ndc"].astype("string").str.strip()
)

if history.duplicated(["package_ndc", "recall_number"]).any():
    raise ValueError("Duplicate recall-package relationships")

feature_frames = []

for year in [2022, 2023, 2024, 2025]:

    cutoff = pd.Timestamp(f"{year}-01-01")

    # Require both dates to precede prediction cutoff.
    eligible = history.loc[
        (history["recall_initiation_date"] < cutoff)
        & (history["report_date"] < cutoff)
    ].copy()

    start_1y = cutoff - pd.DateOffset(years=1)
    start_2y = cutoff - pd.DateOffset(years=2)
    start_5y = cutoff - pd.DateOffset(years=5)

    eligible["class_I"] = (
        eligible["classification"] == "Class I"
    ).astype(int)

    eligible["class_II"] = (
        eligible["classification"] == "Class II"
    ).astype(int)

    eligible["class_III"] = (
        eligible["classification"] == "Class III"
    ).astype(int)

    for window, start in [
        ("1y", start_1y),
        ("2y", start_2y),
        ("5y", start_5y)
    ]:
        eligible[f"within_{window}"] = (
            eligible["recall_initiation_date"] >= start
        ).astype(int)

    features = (
        eligible.groupby("package_ndc", as_index=False)
        .agg(
            prior_recall_count=("recall_number", "nunique"),
            prior_class_I_count=("class_I", "sum"),
            prior_class_II_count=("class_II", "sum"),
            prior_class_III_count=("class_III", "sum"),
            recalls_prior_1y=("within_1y", "sum"),
            recalls_prior_2y=("within_2y", "sum"),
            recalls_prior_5y=("within_5y", "sum"),
            most_recent_prior_recall=(
                "recall_initiation_date", "max"
            )
        )
    )

    features["prediction_year"] = year

    features["days_since_last_recall"] = (
        cutoff - features["most_recent_prior_recall"]
    ).dt.days

    features["has_prior_recall"] = (
        features["prior_recall_count"] > 0
    ).astype(int)

    feature_frames.append(features)

corrected_recall_features = pd.concat(
    feature_frames,
    ignore_index=True
)

assert not corrected_recall_features.duplicated(
    ["package_ndc", "prediction_year"]
).any()

print("PHARMAOPT — CORRECTED RECALL FEATURES")
print("=" * 65)

print("Feature rows:", f"{len(corrected_recall_features):,}")
print("Model years:", sorted(
    corrected_recall_features["prediction_year"].unique()
))

display(
    corrected_recall_features.groupby("prediction_year")
    .agg(
        packages_with_prior_recalls=("package_ndc", "size"),
        total_prior_recall_counts=("prior_recall_count", "sum")
    )
)

print("\nCorrected features constructed in memory.")
print("No original datasets or models modified.")

In [ ]:
# CELL 44 — CORRECTED RECALL FEATURE VALIDATION
# Read-only. Original files and models remain unchanged.

import pandas as pd
import numpy as np

feature_cols = [
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",
    "most_recent_prior_recall",
    "days_since_last_recall",
    "has_prior_recall",
]

keys = ["package_ndc", "prediction_year"]

original = pd.read_csv(
    SOURCE,
    usecols=keys + ["shortage_target"] + feature_cols,
    dtype={"package_ndc": "string"},
    low_memory=False,
)

original["package_ndc"] = original["package_ndc"].str.strip()

assert len(original) == 762907
assert not original.duplicated(keys).any()

rebuilt = corrected_recall_features[
    keys + feature_cols
].copy()

rebuilt["package_ndc"] = rebuilt["package_ndc"].astype("string").str.strip()

validated = original.merge(
    rebuilt,
    on=keys,
    how="left",
    validate="one_to_one",
    suffixes=("_old", "_new"),
)

count_cols = [
    c for c in feature_cols
    if c not in ["most_recent_prior_recall", "days_since_last_recall"]
]

for col in count_cols:
    validated[f"{col}_new"] = (
        validated[f"{col}_new"].fillna(0).astype(int)
    )

print("PHARMAOPT — CORRECTED FEATURE VALIDATION")
print("=" * 65)

print("Original rows:", len(original))
print("Validated rows:", len(validated))
print("Original positives:", int(original["shortage_target"].sum()))
print("Validated positives:", int(validated["shortage_target"].sum()))

assert len(validated) == len(original)
assert not validated.duplicated(keys).any()
assert validated["shortage_target"].equals(original["shortage_target"])

# Count reconciliation
difference = (
    validated["prior_recall_count_old"]
    - validated["prior_recall_count_new"]
)

print("\nRECALL COUNT RECONCILIATION")
print("Affected package-years:", int(difference.ne(0).sum()))
print("Total count reduction:", int(difference.sum()))
print("Counts increasing:", int(difference.lt(0).sum()))

assert int(difference.ne(0).sum()) == 735
assert int(difference.sum()) == 898
assert int(difference.lt(0).sum()) == 0

# Check corrected recall dates
cutoffs = pd.to_datetime(
    validated["prediction_year"].astype(str) + "-01-01"
)

corrected_dates = pd.to_datetime(
    validated["most_recent_prior_recall_new"],
    errors="coerce",
)

future_dates = (
    corrected_dates.notna()
    & (corrected_dates >= cutoffs)
)

print("\nTEMPORAL CHECK")
print("Dates on or after cutoff:", int(future_dates.sum()))

assert int(future_dates.sum()) == 0

# Compare the other corrected features
comparison_rows = []

for col in count_cols:
    old = pd.to_numeric(validated[f"{col}_old"], errors="coerce")
    new = pd.to_numeric(validated[f"{col}_new"], errors="coerce")

    comparison_rows.append({
        "feature": col,
        "changed_rows": int(old.ne(new).sum()),
        "old_total": int(old.sum()),
        "new_total": int(new.sum()),
        "total_reduction": int((old - new).sum()),
    })

feature_comparison = pd.DataFrame(comparison_rows)

print("\nFEATURE-LEVEL COMPARISON")
display(feature_comparison)

print("\nVALIDATION PASSED")
print("No original files or models modified.")

In [ ]:
# CELL 45 — RECALL FEATURE POPULATION RECONCILIATION
# No files modified. No model training.

keys = ["package_ndc", "prediction_year"]

all_rebuilt = corrected_recall_features[
    keys + ["prior_recall_count"]
].copy()

model_keys = original[keys].copy()

reconciliation = all_rebuilt.merge(
    model_keys,
    on=keys,
    how="left",
    indicator=True,
    validate="one_to_one"
)

outside_model = reconciliation.loc[
    reconciliation["_merge"] == "left_only"
]

inside_model = reconciliation.loc[
    reconciliation["_merge"] == "both"
]

print("PHARMAOPT — POPULATION RECONCILIATION")
print("=" * 60)

print(
    "All rebuilt recall counts:",
    int(all_rebuilt["prior_recall_count"].sum())
)

print(
    "Counts within modeling population:",
    int(inside_model["prior_recall_count"].sum())
)

print(
    "Counts outside modeling population:",
    int(outside_model["prior_recall_count"].sum())
)

print(
    "Outside-model package-year rows:",
    len(outside_model)
)

assert (
    all_rebuilt["prior_recall_count"].sum()
    ==
    inside_model["prior_recall_count"].sum()
    + outside_model["prior_recall_count"].sum()
)

assert int(
    inside_model["prior_recall_count"].sum()
) == 48937

assert int(
    outside_model["prior_recall_count"].sum()
) == 923

print("\nPOPULATION RECONCILIATION PASSED")
print("Corrected features are ready for versioned export.")

In [ ]:
# CELL 46 — EXPORT CORRECTED RECALL MODELING PANEL
# Creates a separate versioned dataset.
# Does NOT overwrite original files or retrain models.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import numpy as np

print("PHARMAOPT — VERSIONED RECALL CORRECTION")
print("=" * 65)

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\jpiza\PharmaOpt")

SOURCE = (
    PROJECT_ROOT
    / "outputs"
    / "cms_strict_review_v01"
    / "pharmaopt_integrated_package_year_v05.csv"
)

EXPORT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "recall_timing_corrected_v01"
)

EXPORT_FILE = (
    EXPORT_DIR
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

MANIFEST_FILE = EXPORT_DIR / "validation_manifest_v01.json"

if not SOURCE.is_file():
    raise FileNotFoundError(f"Original panel not found: {SOURCE}")

if EXPORT_FILE.exists() or MANIFEST_FILE.exists():
    raise FileExistsError(
        "A versioned export already exists. "
        "Existing outputs will not be overwritten."
    )

# ------------------------------------------------------------
# 2. VERIFY PREVIOUS VALIDATION IS AVAILABLE
# ------------------------------------------------------------

if "corrected_recall_features" not in globals():
    raise RuntimeError(
        "corrected_recall_features is missing. Run Cell 43 first."
    )

keys = ["package_ndc", "prediction_year"]

feature_cols = [
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",
    "most_recent_prior_recall",
    "days_since_last_recall",
    "has_prior_recall",
]

count_cols = [
    c for c in feature_cols
    if c not in [
        "most_recent_prior_recall",
        "days_since_last_recall",
    ]
]

# ------------------------------------------------------------
# 3. LOAD ORIGINAL 88-COLUMN PANEL
# ------------------------------------------------------------

full_panel = pd.read_csv(
    SOURCE,
    dtype={"package_ndc": "string"},
    low_memory=False,
)

full_panel["package_ndc"] = (
    full_panel["package_ndc"].str.strip()
)

assert len(full_panel) == 762_907
assert len(full_panel.columns) == 88
assert int(full_panel["shortage_target"].sum()) == 426
assert not full_panel.duplicated(keys).any()
assert full_panel[keys].notna().all().all()

original_columns = full_panel.columns.tolist()

print("Original panel loaded.")
print("Rows:", f"{len(full_panel):,}")
print("Columns:", len(full_panel.columns))

# ------------------------------------------------------------
# 4. PREPARE CORRECTED FEATURES
# ------------------------------------------------------------

rebuilt = corrected_recall_features[
    keys + feature_cols
].copy()

rebuilt["package_ndc"] = (
    rebuilt["package_ndc"]
    .astype("string")
    .str.strip()
)

assert not rebuilt.duplicated(keys).any()

# Remove old recall features, then merge corrections.
corrected_panel = (
    full_panel.drop(columns=feature_cols)
    .merge(
        rebuilt,
        on=keys,
        how="left",
        validate="one_to_one",
        sort=False,
        indicator=True,
    )
)

# No eligible history means zero recall counts.
for col in count_cols:
    corrected_panel[col] = (
        corrected_panel[col]
        .fillna(0)
        .astype("int64")
    )

corrected_panel["most_recent_prior_recall"] = (
    pd.to_datetime(
        corrected_panel["most_recent_prior_recall"],
        errors="coerce",
    )
)

corrected_panel["days_since_last_recall"] = (
    pd.to_numeric(
        corrected_panel["days_since_last_recall"],
        errors="coerce",
    )
)

# Remove merge indicator and restore all original columns.
corrected_panel = corrected_panel[original_columns]

# ------------------------------------------------------------
# 5. VALIDATE MODEL POPULATION AND LABELS
# ------------------------------------------------------------

assert len(corrected_panel) == len(full_panel)
assert corrected_panel.columns.tolist() == original_columns
assert not corrected_panel.duplicated(keys).any()

original_labels = full_panel[
    keys + ["shortage_target"]
]

corrected_labels = corrected_panel[
    keys + ["shortage_target"]
]

label_check = original_labels.merge(
    corrected_labels,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_original", "_corrected"),
)

assert len(label_check) == len(full_panel)
assert label_check["_merge"].eq("both").all()

assert (
    label_check["shortage_target_original"]
    == label_check["shortage_target_corrected"]
).all()

assert int(corrected_panel["shortage_target"].sum()) == 426

# ------------------------------------------------------------
# 6. VERIFY RECALL COUNT RECONCILIATION
# ------------------------------------------------------------

comparison = full_panel[
    keys + feature_cols
].merge(
    corrected_panel[keys + feature_cols],
    on=keys,
    how="inner",
    validate="one_to_one",
    suffixes=("_old", "_new"),
)

assert len(comparison) == len(full_panel)

difference = (
    comparison["prior_recall_count_old"]
    - comparison["prior_recall_count_new"]
)

affected_rows = int(difference.ne(0).sum())
count_reduction = int(difference.sum())
count_increases = int(difference.lt(0).sum())

assert affected_rows == 735
assert count_reduction == 898
assert count_increases == 0
assert int(corrected_panel["prior_recall_count"].sum()) == 48_937

# Check every corrected count is no greater than its original.
for col in count_cols:
    old = pd.to_numeric(
        comparison[f"{col}_old"], errors="coerce"
    ).fillna(0)

    new = pd.to_numeric(
        comparison[f"{col}_new"], errors="coerce"
    ).fillna(0)

    assert (new <= old).all(), f"Unexpected count increase: {col}"

# ------------------------------------------------------------
# 7. VERIFY HISTORICAL DATE CUTOFFS
# ------------------------------------------------------------

cutoffs = pd.to_datetime(
    corrected_panel["prediction_year"].astype(str)
    + "-01-01"
)

recall_dates = corrected_panel["most_recent_prior_recall"]

invalid_dates = (
    recall_dates.notna()
    & (recall_dates >= cutoffs)
)

assert int(invalid_dates.sum()) == 0

# Check recall recency values.
expected_days = (cutoffs - recall_dates).dt.days

actual_days = corrected_panel["days_since_last_recall"]

assert (
    expected_days.fillna(-1).to_numpy()
    == actual_days.fillna(-1).to_numpy()
).all()

# ------------------------------------------------------------
# 8. VERIFY NON-RECALL COLUMNS WERE PRESERVED
# ------------------------------------------------------------

untouched_cols = [
    c for c in original_columns
    if c not in feature_cols
]

# Merge by identifiers to guard against row-order changes.
preservation = full_panel[
    keys + [
        c for c in untouched_cols if c not in keys
    ]
].merge(
    corrected_panel[
        keys + [
            c for c in untouched_cols if c not in keys
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one",
    suffixes=("_old", "_new"),
)

assert len(preservation) == len(full_panel)

for col in untouched_cols:
    if col in keys:
        continue

    old = preservation[f"{col}_old"]
    new = preservation[f"{col}_new"]

    assert old.equals(new), (
        f"Unexpected modification to original column: {col}"
    )

print("\nALL IN-MEMORY VALIDATIONS PASSED")
print("Affected package-years:", affected_rows)
print("Recall-count reduction:", count_reduction)
print("Invalid recall dates:", int(invalid_dates.sum()))

# ------------------------------------------------------------
# 9. SAVE TO NEW VERSIONED OUTPUTS
# ------------------------------------------------------------

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Prevent overwriting files created since the earlier check.
if EXPORT_FILE.exists() or MANIFEST_FILE.exists():
    raise FileExistsError(
        "Versioned output already exists. No overwrite permitted."
    )

corrected_panel.to_csv(
    EXPORT_FILE,
    index=False,
    date_format="%Y-%m-%d",
)

# ------------------------------------------------------------
# 10. REOPEN AND VERIFY SAVED CSV
# ------------------------------------------------------------

saved = pd.read_csv(
    EXPORT_FILE,
    usecols=keys + ["shortage_target", "prior_recall_count"],
    dtype={"package_ndc": "string"},
    low_memory=False,
)

assert len(saved) == 762_907
assert not saved.duplicated(keys).any()
assert int(saved["shortage_target"].sum()) == 426
assert int(saved["prior_recall_count"].sum()) == 48_937

# Calculate source/output fingerprints.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

# ------------------------------------------------------------
# 11. CREATE VALIDATION MANIFEST
# ------------------------------------------------------------

manifest = {
    "project": "PharmaOpt",
    "version": "recall_timing_corrected_v01",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(SOURCE),
    "source_sha256": sha256_file(SOURCE),
    "output_file": str(EXPORT_FILE),
    "output_sha256": sha256_file(EXPORT_FILE),
    "rows": 762907,
    "columns": 88,
    "positive_labels": 426,
    "affected_package_years": affected_rows,
    "recall_count_reduction": count_reduction,
    "corrected_recall_total": 48937,
    "timing_rule": (
        "recall_initiation_date < January 1 prediction year "
        "AND report_date < January 1 prediction year"
    ),
    "corrected_features": feature_cols,
    "limitations": [
        "Report date is a proxy for historical availability.",
        "FDA historical records may have been revised.",
        "Shortage-history features require separate validation.",
        "Shortage target definition remains provisional.",
        "2025 evaluation data has previously been inspected.",
    ],
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 65)
print("PHARMAOPT — VERSIONED RECALL EXPORT COMPLETE")
print("=" * 65)

print("Saved dataset:", EXPORT_FILE)
print("Saved manifest:", MANIFEST_FILE)
print("Rows:", f"{len(corrected_panel):,}")
print("Columns:", len(corrected_panel.columns))
print("Shortage labels:", int(corrected_panel["shortage_target"].sum()))
print("Corrected recall total:", int(corrected_panel["prior_recall_count"].sum()))
print("Original dataset preserved: YES")
print("Existing models modified: NO")
print("\nEXPORT COMPLETE — VALIDATION PASSED")

In [ ]:
# CELL 47 — VERIFY VERSIONED RECALL EXPORT

from pathlib import Path
import json
import pandas as pd

folder = Path(r"C:\Users\jpiza\PharmaOpt\outputs\recall_timing_corrected_v01")

csv_file = folder / "pharmaopt_package_year_recall_timing_v01.csv"
manifest_file = folder / "validation_manifest_v01.json"

print("PHARMAOPT — EXPORT VERIFICATION")
print("=" * 60)

print("CSV exists:", csv_file.is_file())
print("Manifest exists:", manifest_file.is_file())

if csv_file.is_file():
    saved = pd.read_csv(
        csv_file,
        usecols=["package_ndc", "prediction_year",
                 "shortage_target", "prior_recall_count"],
        dtype={"package_ndc": "string"},
    )

    assert len(saved) == 762_907
    assert saved["shortage_target"].sum() == 426
    assert saved["prior_recall_count"].sum() == 48_937
    assert not saved.duplicated(
        ["package_ndc", "prediction_year"]
    ).any()

    print("Saved CSV validation: PASSED")
else:
    print("Saved CSV validation: NOT YET AVAILABLE")

if manifest_file.is_file():
    with open(manifest_file, encoding="utf-8") as f:
        manifest = json.load(f)

    print("Manifest version:", manifest["version"])
    print("Manifest validation: PASSED")
else:
    print("Manifest validation: NOT YET AVAILABLE")

In [ ]:
# CELL 48 — SHORTAGE-HISTORY FEATURE INVENTORY
# Read-only: inspect source code and panel columns.

from pathlib import Path
import json
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

source = (
    root / "outputs" / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

notebook_path = (
    root / "Notebooks"
    / "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
)

assert source.is_file()
assert notebook_path.is_file()

columns = pd.read_csv(source, nrows=0).columns.tolist()

shortage_columns = [
    c for c in columns
    if "shortage" in c.lower()
]

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

search_terms = [
    "package_last_shortage",
    "product_last_shortage",
    "generic_last_shortage",
    "labeler_last_shortage",
    "shortage_initial",
    "initial_posting",
    "shortage_start",
]

print("PHARMAOPT — SHORTAGE-HISTORY AUDIT")
print("=" * 65)

print("\nSHORTAGE-RELATED PANEL COLUMNS")
for col in shortage_columns:
    print(" ", col)

print("\nSOURCE CODE REFERENCES")

for i, cell in enumerate(notebook["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()

    matches = [
        (line_no, line.strip())
        for line_no, line in enumerate(lines, start=1)
        if any(term.lower() in line.lower() for term in search_terms)
    ]

    if matches:
        print(f"\nCELL {i}")
        for line_no, line in matches[:20]:
            print(f"  Line {line_no}: {line}")

print("\nRead-only inspection complete.")

In [ ]:
# CELL 49 — SHORTAGE-HISTORY CONSTRUCTION AUDIT
# Read-only: no database queries, file changes, or retraining.

from pathlib import Path
import json

root = Path(r"C:\Users\jpiza\PharmaOpt")

notebook_path = (
    root
    / "Notebooks"
    / "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
)

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

print("PHARMAOPT — SHORTAGE-HISTORY CONSTRUCTION")
print("=" * 65)

for cell_number in [14, 17, 19]:

    cell = notebook["cells"][cell_number - 1]

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    print("\n" + "=" * 65)
    print(f"NOTEBOOK 11 — CELL {cell_number}")
    print("=" * 65)

    # Display all source lines with line numbers.
    for line_number, line in enumerate(
        source.splitlines(), start=1
    ):
        print(f"{line_number:03d}: {line}")

print("\nRead-only inspection complete.")

In [ ]:
# CELL 50 — SHORTAGE-HISTORY INTEGRITY AUDIT
# Read-only. No database changes, exports, or model training.

import pandas as pd
import numpy as np
from pathlib import Path

source = (
    Path(r"C:\Users\jpiza\PharmaOpt")
    / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

assert source.is_file()

cols = [
    "prediction_year",
    "shortage_target",
    "package_prior_shortages",
    "product_prior_shortages",
    "generic_prior_shortages",
    "labeler_prior_shortages",
    "generic_shortages_prior_1y",
    "labeler_shortages_prior_1y",
    "package_shortages_prior_1y",
    "product_shortages_prior_1y",
    "package_has_shortage_history",
    "product_has_shortage_history",
    "generic_has_shortage_history",
    "labeler_has_shortage_history",
    "package_last_shortage",
    "product_last_shortage",
    "generic_last_shortage",
    "labeler_last_shortage",
    "package_days_since_shortage",
    "product_days_since_shortage",
    "generic_days_since_shortage",
    "labeler_days_since_shortage",
    "product_shortage_pressure",
    "generic_shortage_pressure",
    "labeler_shortage_pressure",
]

df = pd.read_csv(source, usecols=cols, low_memory=False)

assert len(df) == 762_907
assert int(df["shortage_target"].sum()) == 426

cutoffs = pd.to_datetime(
    df["prediction_year"].astype(str) + "-01-01"
)

problems = []

# 1. Verify historical dates and recency arithmetic.
for level in ["package", "product", "generic", "labeler"]:

    dates = pd.to_datetime(
        df[f"{level}_last_shortage"],
        errors="coerce"
    )

    days = pd.to_numeric(
        df[f"{level}_days_since_shortage"],
        errors="coerce"
    )

    future = int((dates.notna() & (dates >= cutoffs)).sum())

    expected = (cutoffs - dates).dt.days

    mismatch = int(
        (~(expected.eq(days) | (expected.isna() & days.isna()))).sum()
    )

    problems.append({
        "check": f"{level}_history_dates",
        "future_or_cutoff_dates": future,
        "recency_mismatches": mismatch,
    })

# 2. Verify binary history flags.
flag_results = []

for level in ["package", "product", "generic", "labeler"]:

    expected = (
        df[f"{level}_prior_shortages"] > 0
    ).astype(int)

    actual = df[f"{level}_has_shortage_history"]

    flag_results.append({
        "level": level,
        "flag_mismatches": int((expected != actual).sum())
    })

# 3. Verify recent counts are not greater than all-time counts.
window_results = []

for level in ["package", "product", "generic", "labeler"]:

    recent_col = f"{level}_shortages_prior_1y"
    total_col = f"{level}_prior_shortages"

    if recent_col not in df.columns:
        continue

    window_results.append({
        "level": level,
        "recent_exceeds_total": int(
            (df[recent_col] > df[total_col]).sum()
        )
    })

# 4. Check normalized supply-pressure values.
pressure_results = []

for level in ["product", "generic", "labeler"]:

    values = pd.to_numeric(
        df[f"{level}_shortage_pressure"],
        errors="coerce"
    )

    pressure_results.append({
        "level": level,
        "negative_values": int((values < 0).sum()),
        "missing_values": int(values.isna().sum()),
        "infinite_values": int(np.isinf(values).sum()),
    })

print("PHARMAOPT — SHORTAGE-HISTORY INTEGRITY AUDIT")
print("=" * 65)

print("\nHISTORICAL DATE AND RECENCY CHECKS")
display(pd.DataFrame(problems))

print("\nHISTORY FLAG CHECKS")
display(pd.DataFrame(flag_results))

print("\nONE-YEAR WINDOW CHECKS")
display(pd.DataFrame(window_results))

print("\nSUPPLY-PRESSURE CHECKS")
display(pd.DataFrame(pressure_results))

all_checks_pass = (
    all(x["future_or_cutoff_dates"] == 0 and x["recency_mismatches"] == 0
        for x in problems)
    and all(x["flag_mismatches"] == 0 for x in flag_results)
    and all(x["recent_exceeds_total"] == 0 for x in window_results)
    and all(
        x["negative_values"] == 0
        and x["missing_values"] == 0
        and x["infinite_values"] == 0
        for x in pressure_results
    )
)

print(
    "\nINTEGRITY RESULT:",
    "PASSED" if all_checks_pass else "REQUIRES INVESTIGATION"
)

print("Original files and models unchanged.")

In [ ]:
# CELL 51 — REMAINING FEATURE DEPENDENCY INVENTORY
# Read-only. No database queries or file modifications.

from pathlib import Path
import json
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

panel_path = (
    root / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

notebook_path = (
    root / "Notebooks"
    / "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
)

columns = pd.read_csv(panel_path, nrows=0).columns.tolist()

terms = [
    "cms",
    "marketing",
    "listing_expiration",
    "product_age",
    "active_packages",
    "demand",
]

remaining_columns = [
    c for c in columns
    if any(term in c.lower() for term in terms)
]

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

print("PHARMAOPT — REMAINING FEATURE AUDIT")
print("=" * 65)

print("\nRELEVANT PANEL COLUMNS")
for col in remaining_columns:
    print(" ", col)

print("\nNOTEBOOK SOURCE REFERENCES")

for i, cell in enumerate(notebook["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    matches = [
        (n, line.strip())
        for n, line in enumerate(lines, start=1)
        if any(term in line.lower() for term in terms)
    ]

    if matches:
        print(f"\nCELL {i}: {len(matches)} matching lines")
        for n, line in matches[:12]:
            print(f"  {n}: {line}")

print("\nRead-only dependency inventory complete.")

In [ ]:
# CELL 52 — CMS AND FDA MARKETING DEPENDENCY AUDIT
# Read-only. No exports or model changes.

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

path = (
    root / "Notebooks"
    / "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
)

with open(path, encoding="utf-8") as f:
    notebook = json.load(f)

sections = {
    10: "FDA marketing-date extraction",
    14: "Historical active-package population",
    22: "CMS feature import and year mapping",
    23: "CMS-to-package integration",
    26: "Model feature selection",
    27: "Existing timing limitations",
}

print("PHARMAOPT — REMAINING FEATURE TIMING REVIEW")
print("=" * 65)

for number, title in sections.items():

    cell = notebook["cells"][number - 1]

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    print(f"\n{'=' * 65}")
    print(f"CELL {number}: {title}")
    print("=" * 65)

    if number == 14:
        # Only the population construction section.
        selected = list(enumerate(lines[:66], start=1))

    else:
        selected = list(enumerate(lines, start=1))

    for line_number, line in selected:
        print(f"{line_number:03d}: {line}")

print("\nRead-only inspection complete.")

In [ ]:
# CELL 53 — FDA LISTING EXPIRATION SENSITIVITY AUDIT
# Read-only. No dataset changes or model training.

from pathlib import Path
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

source = (
    root
    / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

cols = [
    "package_ndc",
    "prediction_year",
    "shortage_target",
    "marketing_start_date",
    "marketing_end_date",
    "listing_expiration_date",
]

df = pd.read_csv(
    source,
    usecols=cols,
    dtype={"package_ndc": "string"},
    low_memory=False,
)

assert len(df) == 762_907
assert int(df["shortage_target"].sum()) == 426

cutoffs = pd.to_datetime(
    df["prediction_year"].astype(str) + "-01-01"
)

for col in [
    "marketing_start_date",
    "marketing_end_date",
    "listing_expiration_date",
]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

expired_before_cutoff = (
    df["listing_expiration_date"].notna()
    & (df["listing_expiration_date"] < cutoffs)
)

start_after_cutoff = (
    df["marketing_start_date"].notna()
    & (df["marketing_start_date"] > cutoffs)
)

end_before_cutoff = (
    df["marketing_end_date"].notna()
    & (df["marketing_end_date"] < cutoffs)
)

df["expired_listing_before_cutoff"] = (
    expired_before_cutoff
)

summary = (
    df.groupby("prediction_year")
    .agg(
        model_rows=("package_ndc", "size"),
        expired_listing_rows=(
            "expired_listing_before_cutoff", "sum"
        ),
        positive_labels_on_expired_listings=(
            "shortage_target",
            lambda x: int(
                (
                    x == 1
                ).sum()
            )
        ),
    )
)

# Calculate positives specifically within expired rows.
expired_positives = (
    df.loc[expired_before_cutoff]
    .groupby("prediction_year")["shortage_target"]
    .sum()
)

summary["positive_labels_on_expired_listings"] = (
    expired_positives
    .reindex(summary.index, fill_value=0)
    .astype(int)
)

summary["expired_listing_pct"] = (
    100 * summary["expired_listing_rows"]
    / summary["model_rows"]
)

print("PHARMAOPT — LISTING EXPIRATION AUDIT")
print("=" * 65)

display(summary)

print("\nSTRUCTURAL POPULATION CHECKS")
print("Marketing starts after cutoff:", int(start_after_cutoff.sum()))
print("Marketing ends before cutoff:", int(end_before_cutoff.sum()))
print("Missing listing expiration dates:",
      int(df["listing_expiration_date"].isna().sum()))

print("\nNo rows removed. Original datasets unchanged.")

In [ ]:
# CELL 54 — LISTING EXPIRATION SENSITIVITY
# Read-only. No exports or model training.

import pandas as pd
from pathlib import Path

source = (
    Path(r"C:\Users\jpiza\PharmaOpt")
    / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

df = pd.read_csv(
    source,
    usecols=[
        "package_ndc",
        "prediction_year",
        "shortage_target",
        "listing_expiration_date"
    ],
    dtype={"package_ndc": "string"},
    low_memory=False
)

cutoffs = pd.to_datetime(
    df["prediction_year"].astype(str) + "-01-01"
)

expiration = pd.to_datetime(
    df["listing_expiration_date"],
    errors="coerce"
)

exclude = expiration.notna() & (expiration < cutoffs)

sensitivity = (
    df.assign(excluded=exclude)
    .groupby("prediction_year")
    .agg(
        original_rows=("package_ndc", "size"),
        excluded_rows=("excluded", "sum"),
        original_positives=("shortage_target", "sum")
    )
)

excluded_positives = (
    df.loc[exclude]
    .groupby("prediction_year")["shortage_target"]
    .sum()
)

sensitivity["excluded_positives"] = (
    excluded_positives.reindex(
        sensitivity.index, fill_value=0
    ).astype(int)
)

sensitivity["remaining_rows"] = (
    sensitivity["original_rows"]
    - sensitivity["excluded_rows"]
)

sensitivity["remaining_positives"] = (
    sensitivity["original_positives"]
    - sensitivity["excluded_positives"]
)

print("PHARMAOPT — POPULATION SENSITIVITY")
print("=" * 65)
display(sensitivity)

assert int(sensitivity["excluded_rows"].sum()) == 12
assert int(sensitivity["excluded_positives"].sum()) == 0
assert int(sensitivity["remaining_rows"].sum()) == 762_895
assert int(sensitivity["remaining_positives"].sum()) == 426

print("\nSENSITIVITY CHECK PASSED")
print("Original panel preserved.")

In [ ]:
# CELL 55 — PRE-RETRAINING READINESS REVIEW
# Read-only. No model training or file changes.

from pathlib import Path
import json
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

folder = (
    root / "outputs" / "recall_timing_corrected_v01"
)

panel_file = (
    folder / "pharmaopt_package_year_recall_timing_v01.csv"
)

manifest_file = folder / "validation_manifest_v01.json"

assert panel_file.is_file()
assert manifest_file.is_file()

with open(manifest_file, encoding="utf-8") as f:
    manifest = json.load(f)

panel = pd.read_csv(
    panel_file,
    usecols=[
        "package_ndc",
        "prediction_year",
        "shortage_target",
        "prior_recall_count",
        "cms_data_year",
        "cms_demand_match",
    ],
    dtype={"package_ndc": "string"},
    low_memory=False,
)

assert len(panel) == 762_907
assert panel["shortage_target"].sum() == 426
assert not panel.duplicated(
    ["package_ndc", "prediction_year"]
).any()

matched = panel["cms_demand_match"].eq(1)

assert (
    panel.loc[matched, "prediction_year"]
    - panel.loc[matched, "cms_data_year"]
).eq(3).all()

summary = (
    panel.groupby("prediction_year")
    .agg(
        package_years=("package_ndc", "size"),
        positive_labels=("shortage_target", "sum"),
        cms_matched_rows=("cms_demand_match", "sum"),
        corrected_recall_counts=("prior_recall_count", "sum"),
    )
)

print("PHARMAOPT — PRE-RETRAINING READINESS")
print("=" * 65)

display(summary)

print("\nVERIFIED")
print("- Corrected recall feature panel saved")
print("- Original package-year population preserved")
print("- Original outcome labels preserved")
print("- Recall timing discrepancy reconciled")
print("- Shortage-history integrity checks passed")
print("- Listing-expiration sensitivity passed")
print("- CMS three-year mapping passed")

print("\nREMAINING LIMITATIONS")
print("- CMS actual release dates not independently verified")
print("- Historical FDA source availability not established")
print("- FDA shortage target remains provisional")
print("- 2025 was previously inspected during development")
print("- Corrected models have not yet been evaluated")

print("\nREADINESS REVIEW COMPLETE")
print("Status: READY FOR CONTROLLED RESEARCH RETRAINING")
print("Status: NOT READY FOR PRODUCTION FORECASTING")

In [ ]:
# CELL 56 — FDA OUTCOME DEFINITION INVENTORY
# Read-only. Does not modify notebooks, datasets, or models.

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

notebook_path = (
    root / "Notebooks"
    / "15_pharmaopt_fda_outcome_definition.ipynb"
)

if not notebook_path.is_file():
    raise FileNotFoundError(notebook_path)

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

terms = [
    "discontinu",
    "shortage_target",
    "outcome",
    "initial_posting_date",
    "status",
    "label_definition",
    "positive",
]

print("PHARMAOPT — FDA OUTCOME DEFINITION INVENTORY")
print("=" * 65)

for number, cell in enumerate(notebook["cells"], start=1):
    source = "".join(cell.get("source", []))

    if not source.strip():
        continue

    matches = [
        line.strip()
        for line in source.splitlines()
        if any(term in line.lower() for term in terms)
    ]

    if matches:
        print(
            f"Cell {number:02d} | "
            f"{cell['cell_type']} | "
            f"{len(matches)} matching lines"
        )

        for line in matches[:5]:
            print(" ", line[:150])

print("\nInventory complete.")

In [ ]:
# CELL 57 — FDA OUTCOME CONCLUSION REVIEW
# Read-only. No model training or file changes.

import json
from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")
path = root / "Notebooks" / "15_pharmaopt_fda_outcome_definition.ipynb"

with open(path, encoding="utf-8") as f:
    notebook = json.load(f)

print("PHARMAOPT — OUTCOME DEFINITION REVIEW")
print("=" * 65)

for number in [4, 5, 6, 10, 12, 14, 15]:
    cell = notebook["cells"][number - 1]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 65}")
    print(f"CELL {number}")
    print("=" * 65)

    for line_number, line in enumerate(source.splitlines(), 1):
        print(f"{line_number:03d}: {line}")

print("\nREVIEW COMPLETE")

In [ ]:
# CELL 58 — FDA OUTCOME FILE READINESS
# Read-only. No retraining or file changes.

from pathlib import Path
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

panel_path = (
    root / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

labels_path = (
    root / "outputs"
    / "fda_outcome_definition"
    / "pharmaopt_fda_outcome_labels_v01.csv"
)

print("PHARMAOPT — OUTCOME READINESS CHECK")
print("=" * 60)

print("Corrected panel exists:", panel_path.is_file())
print("Outcome labels exist:", labels_path.is_file())

assert panel_path.is_file()
assert labels_path.is_file()

panel = pd.read_csv(
    panel_path,
    usecols=["package_ndc", "prediction_year", "shortage_target"],
    dtype={"package_ndc": "string"},
    low_memory=False,
)

labels = pd.read_csv(
    labels_path,
    dtype={"package_ndc": "string"},
    low_memory=False,
)

keys = ["package_ndc", "prediction_year"]

for df in [panel, labels]:
    df["package_ndc"] = (
        df["package_ndc"].astype("string").str.strip()
    )

assert not panel.duplicated(keys).any()
assert not labels.duplicated(keys).any()

print("\nAVAILABLE OUTCOME COLUMNS")
print(labels.columns.tolist())

required = [
    "non_discontinuation_event_candidate",
    "discontinuation_notice_candidate",
]

missing = sorted(set(required) - set(labels.columns))
assert not missing, f"Missing outcome columns: {missing}"

joined = panel.merge(
    labels[keys + required],
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\nJOIN VALIDATION")
print("Corrected panel rows:", len(panel))
print("Matched outcome rows:", int(joined["_merge"].eq("both").sum()))
print("Unmatched outcome rows:", int(joined["_merge"].eq("left_only").sum()))

assert len(joined) == 762_907
assert joined["_merge"].eq("both").all()

print("\nOUTCOME COUNTS BY YEAR")

display(
    joined.groupby("prediction_year")[
        ["shortage_target"] + required
    ].sum()
)

print("\nOUTCOME READINESS CHECK PASSED")
print("No files or models modified.")

In [ ]:
# CELL 59 — FREEZE CORRECTED MODEL EXPERIMENT
# Read-only. Defines the next research experiment.

from pathlib import Path
import json
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

corrected_file = (
    root / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

labels_file = (
    root / "outputs"
    / "fda_outcome_definition"
    / "pharmaopt_fda_outcome_labels_v01.csv"
)

source_notebook = (
    root / "Notebooks"
    / "11_pharmaopt_integrated_pipeline_CMS_STRICT_REVIEW.ipynb"
)

assert corrected_file.is_file()
assert labels_file.is_file()
assert source_notebook.is_file()

with open(source_notebook, encoding="utf-8") as f:
    notebook = json.load(f)

# Extract the exact original feature definitions.
source = "".join(notebook["cells"][25]["source"])
namespace = {}

start = source.find("features_v03_numeric =")
end = source.find('print("Feature definitions loaded successfully.")')

if start == -1 or end == -1:
    raise RuntimeError("Cannot locate original model feature definitions.")

exec(source[start:end], {}, namespace)

features_v04_numeric = namespace["features_v04_numeric"]
features_v04_cat = namespace["features_v04_cat"]

available = pd.read_csv(corrected_file, nrows=0).columns.tolist()

# Preserve the original qualitative feature selection rule.
qualitative_features = [
    c for c in available
    if c.startswith("prior_")
    and (c.endswith("_3y") or c.endswith("_1y"))
]

assert len(qualitative_features) == 16

features_v05_numeric = list(dict.fromkeys(
    features_v04_numeric
    + qualitative_features
    + ["linked_recall_history_record_3y"]
))

features_v05_cat = list(features_v04_cat)

for version, columns in {
    "v04": features_v04_numeric + features_v04_cat,
    "v05": features_v05_numeric + features_v05_cat,
}.items():
    missing = sorted(set(columns) - set(available))
    if missing:
        raise ValueError(f"{version} missing features: {missing}")

EXPERIMENT = {
    "panel": corrected_file,
    "labels": labels_file,
    "primary_target": "shortage_target",
    "secondary_target": "non_discontinuation_event_candidate",
    "training_years": [2022, 2023, 2024],
    "evaluation_year": 2025,
    "evaluation_status": "previously_inspected_development_holdout",
    "v04_numeric": features_v04_numeric,
    "v04_categorical": features_v04_cat,
    "v05_numeric": features_v05_numeric,
    "v05_categorical": features_v05_cat,
}

print("PHARMAOPT — CORRECTED EXPERIMENT CONFIGURATION")
print("=" * 65)

print("Primary outcome:", EXPERIMENT["primary_target"])
print("Secondary outcome:", EXPERIMENT["secondary_target"])
print("Training years:", EXPERIMENT["training_years"])
print("Evaluation year:", EXPERIMENT["evaluation_year"])
print("v0.4 features:", len(features_v04_numeric) + len(features_v04_cat))
print("v0.5 features:", len(features_v05_numeric) + len(features_v05_cat))
print("Qualitative features:", len(qualitative_features))

print("\nEXPERIMENT CONFIGURATION VERIFIED")
print("No files or models modified.")


In [ ]:
# CELL 60 — PREPARE CORRECTED MODEL TRAINING DATA
# Read-only. No model training or file modifications.

import pandas as pd
import numpy as np

print("PHARMAOPT — CORRECTED TRAINING DATA PREPARATION")
print("=" * 65)

if "EXPERIMENT" not in globals():
    raise RuntimeError("Run Cell 59 before continuing.")

keys = ["package_ndc", "prediction_year"]

features_v04 = list(dict.fromkeys(
    EXPERIMENT["v04_numeric"] + EXPERIMENT["v04_categorical"]
))

features_v05 = list(dict.fromkeys(
    EXPERIMENT["v05_numeric"] + EXPERIMENT["v05_categorical"]
))

required_panel_columns = list(dict.fromkeys(
    keys + [EXPERIMENT["primary_target"]] + features_v05
))

# Load only the columns needed for model training.
model_data = pd.read_csv(
    EXPERIMENT["panel"],
    usecols=required_panel_columns,
    dtype={"package_ndc": "string"},
    low_memory=False
)

model_data["package_ndc"] = (
    model_data["package_ndc"].str.strip()
)

# Load existing independently defined secondary labels.
secondary_target = EXPERIMENT["secondary_target"]

labels = pd.read_csv(
    EXPERIMENT["labels"],
    usecols=keys + [secondary_target],
    dtype={"package_ndc": "string"},
    low_memory=False
)

labels["package_ndc"] = (
    labels["package_ndc"].str.strip()
)

assert not model_data.duplicated(keys).any()
assert not labels.duplicated(keys).any()

model_data = model_data.merge(
    labels,
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert model_data["_merge"].eq("both").all()
model_data = model_data.drop(columns="_merge")

# Confirm dataset integrity.
assert len(model_data) == 762_907
assert int(model_data["shortage_target"].sum()) == 426

assert model_data["shortage_target"].isin([0, 1]).all()
assert model_data[secondary_target].isin([0, 1]).all()

# Create chronological training/evaluation split.
train_mask = model_data["prediction_year"].isin(
    EXPERIMENT["training_years"]
)

test_mask = (
    model_data["prediction_year"]
    == EXPERIMENT["evaluation_year"]
)

assert (train_mask | test_mask).all()
assert not (train_mask & test_mask).any()

X_train_v04 = model_data.loc[
    train_mask, features_v04
].copy()

X_test_v04 = model_data.loc[
    test_mask, features_v04
].copy()

X_train_v05 = model_data.loc[
    train_mask, features_v05
].copy()

X_test_v05 = model_data.loc[
    test_mask, features_v05
].copy()

y_train = model_data.loc[
    train_mask, "shortage_target"
].astype(int).copy()

y_test = model_data.loc[
    test_mask, "shortage_target"
].astype(int).copy()

secondary_test = model_data.loc[
    test_mask, secondary_target
].astype(int).copy()

# Check expected split.
assert len(y_train) == 541_240
assert int(y_train.sum()) == 330

assert len(y_test) == 221_667
assert int(y_test.sum()) == 96

assert X_train_v04.index.equals(X_train_v05.index)
assert X_test_v04.index.equals(X_test_v05.index)

print("\nDATASET SUMMARY")
print("Total observations:", f"{len(model_data):,}")
print("Training observations:", f"{len(y_train):,}")
print("Evaluation observations:", f"{len(y_test):,}")

print("\nOUTCOME COUNTS")
print("Training FDA-linked events:", int(y_train.sum()))
print("Evaluation FDA-linked events:", int(y_test.sum()))
print(
    "Evaluation non-discontinuation candidates:",
    int(secondary_test.sum())
)

print("\nFEATURE COUNTS")
print("v0.4:", len(features_v04))
print("v0.5:", len(features_v05))

print("\nCORRECTED TRAINING DATA VERIFIED")
print("Models trained: NO")
print("Original datasets modified: NO")

In [ ]:
# CELL 61 — RETRAIN CORRECTED v0.4 BASELINE
# In-memory training only. No files overwritten.

import time
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

print("PHARMAOPT — CORRECTED v0.4 MODEL TRAINING")
print("=" * 65)

# Confirm the prepared experiment is available.
for name in ["X_train_v04", "X_test_v04", "y_train", "y_test", "EXPERIMENT"]:
    if name not in globals():
        raise RuntimeError(f"Missing {name}. Run Cell 60 first.")

# Reproduce the original v0.4 preprocessing and model.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        EXPERIMENT["v04_numeric"],
    ),
    (
        "categorical",
        categorical_pipeline,
        EXPERIMENT["v04_categorical"],
    ),
])

model_v04_corrected = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2500,
            solver="liblinear",
        ),
    ),
])

# Train corrected baseline.
start = time.perf_counter()

print("Training corrected v0.4...", flush=True)

model_v04_corrected.fit(X_train_v04, y_train)

elapsed = time.perf_counter() - start

print("Training completed.", flush=True)
print("Training duration (seconds):", round(elapsed, 2))

# Score previously inspected 2025 development evaluation set.
score_v04_corrected = model_v04_corrected.predict_proba(
    X_test_v04
)[:, 1]

assert len(score_v04_corrected) == len(y_test)
assert np.isfinite(score_v04_corrected).all()

# Evaluation metrics.
ap_v04_corrected = average_precision_score(
    y_test, score_v04_corrected
)

roc_v04_corrected = roc_auc_score(
    y_test, score_v04_corrected
)

# Deterministic ranking with package NDC as tiebreaker.
package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str)
    .to_numpy()
)

order = np.lexsort((package_ids, -score_v04_corrected))

top500_v04_corrected = int(
    y_test.to_numpy()[order[:500]].sum()
)

print("\nCORRECTED v0.4 RESULTS")
print("-" * 45)

print("Average precision:", round(ap_v04_corrected, 6))
print("ROC-AUC:", round(roc_v04_corrected, 6))
print("FDA-linked events in top 500:", top500_v04_corrected)
print("Evaluation baseline rate:", round(float(y_test.mean()), 6))

print("\nCORRECTED v0.4 TRAINING VERIFIED")
print("Evaluation: previously inspected 2025 development set")
print("Original model files: unchanged")

In [ ]:
# CELL 62 — CORRECTED PHARMAOPT v0.5 RETRAINING
# Independent of v0.4. No existing files overwritten.

import time
import threading
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

print("PHARMAOPT — CORRECTED v0.5 TRAINING", flush=True)
print("=" * 65, flush=True)

# ------------------------------------------------------------
# 1. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required = [
    "X_train_v05",
    "X_test_v05",
    "y_train",
    "y_test",
    "EXPERIMENT",
    "model_data",
    "test_mask",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing variables: {missing}")

print("Training observations:", f"{len(y_train):,}", flush=True)
print("Evaluation observations:", f"{len(y_test):,}", flush=True)
print("Model features:", X_train_v05.shape[1], flush=True)

# ------------------------------------------------------------
# 2. ORIGINAL v0.5 MODEL ARCHITECTURE
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value=-1
    )),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    )),
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        EXPERIMENT["v05_numeric"],
    ),
    (
        "categorical",
        categorical_pipeline,
        EXPERIMENT["v05_categorical"],
    ),
])

model_v05_corrected = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2500,
            solver="liblinear",
        ),
    ),
])

# ------------------------------------------------------------
# 3. TRAIN WITH PROGRESS MESSAGES
# ------------------------------------------------------------

start = time.perf_counter()
stop_event = threading.Event()

def training_heartbeat():
    while not stop_event.wait(30):
        elapsed = time.perf_counter() - start
        print(
            f"v0.5 still training — {elapsed / 60:.1f} minutes",
            flush=True
        )

heartbeat_thread = threading.Thread(
    target=training_heartbeat,
    daemon=True
)

print("\nStarting corrected v0.5 training...", flush=True)

heartbeat_thread.start()

try:
    model_v05_corrected.fit(X_train_v05, y_train)
finally:
    stop_event.set()
    heartbeat_thread.join(timeout=2)

elapsed = time.perf_counter() - start

print("\nTraining completed!", flush=True)
print("Training time (seconds):", round(elapsed, 2))

# ------------------------------------------------------------
# 4. EVALUATE 2025 DEVELOPMENT DATA
# ------------------------------------------------------------

score_v05_corrected = model_v05_corrected.predict_proba(
    X_test_v05
)[:, 1]

assert len(score_v05_corrected) == len(y_test)
assert np.isfinite(score_v05_corrected).all()

ap_v05_corrected = average_precision_score(
    y_test,
    score_v05_corrected
)

roc_v05_corrected = roc_auc_score(
    y_test,
    score_v05_corrected
)

package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str)
    .to_numpy()
)

order = np.lexsort(
    (package_ids, -score_v05_corrected)
)

top500_v05_corrected = int(
    y_test.to_numpy()[order[:500]].sum()
)

# ------------------------------------------------------------
# 5. RESULTS
# ------------------------------------------------------------

print("\nPHARMAOPT — CORRECTED v0.5 RESULTS")
print("=" * 65)

print("Average precision:", round(ap_v05_corrected, 6))
print("ROC-AUC:", round(roc_v05_corrected, 6))
print("FDA-linked events in top 500:", top500_v05_corrected)
print("Baseline event rate:", round(float(y_test.mean()), 6))

print("\nCORRECTED v0.5 TRAINING VERIFIED")
print("Original datasets and model files unchanged.")

In [ ]:
# CELL 63 — v0.5 TRAINING DIAGNOSTICS
# Read-only. No training or file changes.

import numpy as np
import pandas as pd

print("PHARMAOPT — v0.5 TRAINING DIAGNOSTICS")
print("=" * 65)

required = ["X_train_v05", "y_train", "EXPERIMENT"]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

numeric_cols = EXPERIMENT["v05_numeric"]
numeric = X_train_v05[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

summary = pd.DataFrame({
    "feature": numeric_cols,
    "missing_pct": numeric.isna().mean().to_numpy() * 100,
    "minimum": numeric.min().to_numpy(),
    "maximum": numeric.max().to_numpy(),
    "unique_values": numeric.nunique().to_numpy(),
})

print("\nDATA SHAPE")
print("Rows:", len(X_train_v05))
print("Features:", X_train_v05.shape[1])

print("\nMEMORY")
print(
    "Training DataFrame MB:",
    round(X_train_v05.memory_usage(deep=True).sum() / 1024**2, 2)
)

print("\nNUMERIC FEATURE SUMMARY")
display(summary)

print("\nEXTREME VALUES")
print("Infinite values:", int(np.isinf(numeric.to_numpy()).sum()))

print("\nCLASS DISTRIBUTION")
print(y_train.value_counts())

print("\nDIAGNOSTICS COMPLETE")

In [ ]:
# CELL 64 — v0.5 PREPROCESSING DIAGNOSTICS
# Does not train the logistic regression model.

import time
import numpy as np
from sklearn.base import clone

print("PHARMAOPT — v0.5 PREPROCESSING CHECK")
print("=" * 60)

required = ["model_v05_corrected", "X_train_v05"]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

start = time.perf_counter()

processor = clone(
    model_v05_corrected.named_steps["preprocessor"]
)

X_processed = processor.fit_transform(X_train_v05)

elapsed = time.perf_counter() - start

print("Preprocessing seconds:", round(elapsed, 2))
print("Processed shape:", X_processed.shape)
print("Processed matrix type:", type(X_processed).__name__)
print("Processed matrix dtype:", X_processed.dtype)

if hasattr(X_processed, "nnz"):
    print("Nonzero values:", X_processed.nnz)

print("All finite:", bool(np.isfinite(X_processed.data).all())
      if hasattr(X_processed, "data")
      else bool(np.isfinite(X_processed).all()))

print("\nPREPROCESSING CHECK COMPLETE")

In [ ]:
# CELL 65 — CONTROLLED v0.5 SOLVER EXPERIMENT
# Uses existing preprocessing from Cell 64.
# No existing files overwritten.

import time
import warnings
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

print("PHARMAOPT — v0.5 CONTROLLED SOLVER EXPERIMENT")
print("=" * 65, flush=True)

required = [
    "processor", "X_processed", "X_test_v05",
    "y_train", "y_test", "model_data", "test_mask"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

# Train with the same balanced logistic regression objective,
# but change the optimization solver.
candidate = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    max_iter=300,
    tol=1e-4
)

start = time.perf_counter()

print("Starting controlled training...", flush=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    candidate.fit(X_processed, y_train)

elapsed = time.perf_counter() - start

convergence_warnings = [
    str(w.message) for w in caught
    if issubclass(w.category, ConvergenceWarning)
]

print("Training seconds:", round(elapsed, 2))
print("Iterations used:", candidate.n_iter_.tolist())
print("Convergence warnings:", len(convergence_warnings))

for warning in convergence_warnings:
    print("WARNING:", warning)

# Score evaluation data using the same fitted preprocessor.
X_test_processed = processor.transform(X_test_v05)

scores = candidate.predict_proba(X_test_processed)[:, 1]

assert len(scores) == len(y_test)
assert np.isfinite(scores).all()

package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str).to_numpy()
)

order = np.lexsort((package_ids, -scores))

ap = average_precision_score(y_test, scores)
roc = roc_auc_score(y_test, scores)
top500 = int(y_test.to_numpy()[order[:500]].sum())

# Preserve results in memory.
model_v05_lbfgs = Pipeline([
    ("preprocessor", processor),
    ("model", candidate)
])

score_v05_lbfgs = scores

print("\nCONTROLLED v0.5 RESULTS")
print("Average precision:", round(ap, 6))
print("ROC-AUC:", round(roc, 6))
print("FDA-linked events in top 500:", top500)

print("\nCONTROLLED SOLVER EXPERIMENT COMPLETE")
print("Original model configuration and files unchanged.")

In [ ]:
# CELL 66 — v0.5 CONVERGENCE VALIDATION
# Reuses the fitted preprocessor; no files overwritten.

import time
import warnings
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

print("PHARMAOPT — v0.5 CONVERGENCE VALIDATION")
print("=" * 65, flush=True)

required = [
    "X_processed", "X_test_v05", "processor",
    "y_train", "y_test", "model_data", "test_mask"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing variables: {missing}")

candidate_v05_1000 = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    max_iter=1000,
    tol=1e-4,
)

start = time.perf_counter()

print("Training v0.5 with 1,000-iteration limit...", flush=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    candidate_v05_1000.fit(X_processed, y_train)

elapsed = time.perf_counter() - start

warnings_found = [
    str(w.message) for w in caught
    if issubclass(w.category, ConvergenceWarning)
]

scores_v05_1000 = candidate_v05_1000.predict_proba(
    processor.transform(X_test_v05)
)[:, 1]

assert np.isfinite(scores_v05_1000).all()

package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str)
    .to_numpy()
)

order = np.lexsort((package_ids, -scores_v05_1000))
labels = y_test.to_numpy(dtype=int)

ap = average_precision_score(labels, scores_v05_1000)
roc = roc_auc_score(labels, scores_v05_1000)
top500 = int(labels[order[:500]].sum())

print("\nRESULTS")
print("Training seconds:", round(elapsed, 2))
print("Iterations used:", candidate_v05_1000.n_iter_.tolist())
print("Convergence warnings:", len(warnings_found))
print("Average precision:", round(ap, 6))
print("ROC-AUC:", round(roc, 6))
print("FDA-linked events in top 500:", top500)

print("\nCONVERGENCE VALIDATION COMPLETE")
print("Original files unchanged.")

In [ ]:
# CELL 67 — CONVERGED v0.5 OUTCOME SENSITIVITY
# Read-only evaluation. No retraining or file changes.

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

print("PHARMAOPT — v0.5 OUTCOME SENSITIVITY")
print("=" * 65)

required = [
    "scores_v05_1000",
    "secondary_test",
    "y_test",
    "model_data",
    "test_mask",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

scores = np.asarray(scores_v05_1000, dtype=float)

package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str)
    .to_numpy()
)

order = np.lexsort((package_ids, -scores))

outcomes = {
    "All FDA-linked events": y_test.to_numpy(dtype=int),
    "Non-discontinuation candidates": (
        secondary_test.to_numpy(dtype=int)
    ),
}

rows = []

for name, labels in outcomes.items():

    assert len(labels) == len(scores)
    assert np.isfinite(scores).all()

    rows.append({
        "outcome": name,
        "positive_events": int(labels.sum()),
        "positive_rate": float(labels.mean()),
        "average_precision": float(
            average_precision_score(labels, scores)
        ),
        "roc_auc": float(
            roc_auc_score(labels, scores)
        ),
        "top_500_captured": int(
            labels[order[:500]].sum()
        ),
        "top_1000_captured": int(
            labels[order[:1000]].sum()
        ),
    })

outcome_sensitivity = pd.DataFrame(rows)

display(outcome_sensitivity.round(6))

print("\nOUTCOME SENSITIVITY COMPLETE")
print("Model: corrected v0.5, converged lbfgs")
print("Evaluation: 2025 development set")
print("No original files modified.")

In [ ]:
# CELL 68 — RESTORE CORRECTED v0.4 AND COMPARE FDA OUTCOMES
# In-memory training and evaluation. No file overwrites.

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

print("PHARMAOPT — CORRECTED MODEL OUTCOME COMPARISON")
print("=" * 65, flush=True)

required = [
    "EXPERIMENT", "X_train_v04", "X_test_v04",
    "y_train", "y_test", "secondary_test",
    "scores_v05_1000", "model_data", "test_mask"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_v04 = ColumnTransformer([
    ("numeric", numeric_pipe, EXPERIMENT["v04_numeric"]),
    ("categorical", categorical_pipe, EXPERIMENT["v04_categorical"]),
])

model_v04_corrected = Pipeline([
    ("preprocessor", preprocessor_v04),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2500,
        solver="liblinear"
    ))
])

print("Training corrected v0.4 baseline...", flush=True)
start = time.perf_counter()

model_v04_corrected.fit(X_train_v04, y_train)

print(
    "Training seconds:",
    round(time.perf_counter() - start, 2),
    flush=True
)

print(
    "v0.4 iterations used:",
    model_v04_corrected.named_steps["model"].n_iter_.tolist()
)

score_v04_corrected = model_v04_corrected.predict_proba(
    X_test_v04
)[:, 1]

scores_by_model = {
    "v0.4 corrected (liblinear)": score_v04_corrected,
    "v0.5 corrected (lbfgs)": np.asarray(scores_v05_1000),
}

outcomes = {
    "All FDA-linked events": y_test.to_numpy(dtype=int),
    "Non-discontinuation candidates": secondary_test.to_numpy(dtype=int),
}

package_ids = (
    model_data.loc[test_mask, "package_ndc"]
    .astype(str)
    .to_numpy()
)

rows = []

for model_name, scores in scores_by_model.items():
    assert len(scores) == len(package_ids)
    assert np.isfinite(scores).all()

    order = np.lexsort((package_ids, -scores))

    for outcome_name, labels in outcomes.items():
        rows.append({
            "model": model_name,
            "outcome": outcome_name,
            "positives": int(labels.sum()),
            "average_precision": average_precision_score(labels, scores),
            "roc_auc": roc_auc_score(labels, scores),
            "top_500_captured": int(labels[order[:500]].sum()),
            "top_1000_captured": int(labels[order[:1000]].sum())
        })

corrected_outcome_comparison = pd.DataFrame(rows)

print("\nCORRECTED MODEL OUTCOME COMPARISON")
display(corrected_outcome_comparison.round(6))

print("\nCELL 68 COMPLETE")
print("Original files and previous model outputs unchanged.")

In [ ]:
# CELL 69 — SAVE CORRECTED MODEL EXPERIMENT
# Creates new versioned outputs. Does not overwrite prior files.

from pathlib import Path
from datetime import datetime, timezone
import json
import joblib
import numpy as np
import pandas as pd

root = Path(r"C:\Users\jpiza\PharmaOpt")

output_dir = (
    root / "outputs" / "corrected_model_comparison_v01"
)

output_dir.mkdir(parents=True, exist_ok=True)

files = {
    "v04_model": output_dir / "pharmaopt_v04_corrected.joblib",
    "v05_model": output_dir / "pharmaopt_v05_lbfgs_corrected.joblib",
    "scores": output_dir / "pharmaopt_2025_corrected_scores.csv",
    "comparison": output_dir / "corrected_outcome_comparison.csv",
    "manifest": output_dir / "experiment_manifest.json",
}

# Protect previous outputs.
existing = [str(p) for p in files.values() if p.exists()]
if existing:
    raise FileExistsError(f"Outputs already exist: {existing}")

required = [
    "model_v04_corrected",
    "candidate_v05_1000",
    "processor",
    "score_v04_corrected",
    "scores_v05_1000",
    "corrected_outcome_comparison",
    "model_data",
    "test_mask",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

from sklearn.pipeline import Pipeline

model_v05_converged = Pipeline([
    ("preprocessor", processor),
    ("model", candidate_v05_1000),
])

# Restore the package identifiers and outcome labels.
scores_df = model_data.loc[
    test_mask,
    [
        "package_ndc",
        "prediction_year",
        "shortage_target",
        "non_discontinuation_event_candidate",
    ],
].copy().reset_index(drop=True)

scores_df["risk_score_v04"] = np.asarray(
    score_v04_corrected, dtype=float
)

scores_df["risk_score_v05"] = np.asarray(
    scores_v05_1000, dtype=float
)

assert len(scores_df) == 221_667
assert int(scores_df["shortage_target"].sum()) == 96
assert int(
    scores_df["non_discontinuation_event_candidate"].sum()
) == 12

assert np.isfinite(
    scores_df[["risk_score_v04", "risk_score_v05"]]
    .to_numpy()
).all()

# Export fitted models.
joblib.dump(model_v04_corrected, files["v04_model"])
joblib.dump(model_v05_converged, files["v05_model"])

# Export predictions and metrics.
scores_df.to_csv(files["scores"], index=False)

corrected_outcome_comparison.to_csv(
    files["comparison"],
    index=False,
)

manifest = {
    "project": "PharmaOpt",
    "experiment": "corrected_model_comparison_v01",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_panel": str(EXPERIMENT["panel"]),
    "training_years": [2022, 2023, 2024],
    "evaluation_year": 2025,
    "evaluation_status": "previously_inspected_development_holdout",
    "primary_outcome": "FDA-linked event within prediction year",
    "secondary_outcome": "non-discontinuation FDA event candidate",
    "v04_solver": "liblinear",
    "v05_solver": "lbfgs",
    "v05_max_iter": 1000,
    "v05_iterations_used": 526,
    "primary_model_decision": "v0.4",
    "limitations": [
        "Evaluation set previously inspected during development",
        "FDA outcome labels not independently verified as active shortages",
        "FDA records may have been revised retrospectively",
        "CMS release timing not independently verified",
        "Different logistic regression solvers were used",
        "Scores are not calibrated shortage probabilities",
    ],
}

files["manifest"].write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("PHARMAOPT — CORRECTED MODEL EXPERIMENT SAVED")
print("=" * 65)

for name, path in files.items():
    assert path.is_file()
    print(name, ":", path.name)

print("\nMODEL EXPERIMENT EXPORT VERIFIED")
print("Original files preserved.")

In [ ]:
# CELL 70 — SAVED MODEL REPRODUCIBILITY CHECK
# Read-only: does not train models or modify files.

from pathlib import Path
import json
import joblib
import pandas as pd

ROOT = Path(r"C:\Users\jpiza\PharmaOpt")

FOLDER = ROOT / "outputs" / "corrected_model_comparison_v01"

paths = {
    "v0.4 model": FOLDER / "pharmaopt_v04_corrected.joblib",
    "v0.5 model": FOLDER / "pharmaopt_v05_lbfgs_corrected.joblib",
    "scores": FOLDER / "pharmaopt_2025_corrected_scores.csv",
    "comparison": FOLDER / "corrected_outcome_comparison.csv",
    "manifest": FOLDER / "experiment_manifest.json",
}

print("PHARMAOPT — SAVED MODEL REPRODUCIBILITY")
print("=" * 65)

for name, path in paths.items():
    assert path.is_file(), f"Missing {name}: {path}"
    print(f"FOUND: {name}")

# Load only trusted models generated in your PharmaOpt project.
model_v04_saved = joblib.load(paths["v0.4 model"])
model_v05_saved = joblib.load(paths["v0.5 model"])

scores_saved = pd.read_csv(
    paths["scores"],
    dtype={"package_ndc": "string"},
    low_memory=False,
)

with open(paths["manifest"], encoding="utf-8") as f:
    saved_manifest = json.load(f)

assert len(scores_saved) == 221_667
assert scores_saved["package_ndc"].nunique() == 221_667
assert int(scores_saved["shortage_target"].sum()) == 96
assert int(
    scores_saved["non_discontinuation_event_candidate"].sum()
) == 12

assert saved_manifest["primary_model_decision"] == "v0.4"

print("\nLOADED MODELS")
print("v0.4:", type(model_v04_saved).__name__)
print("v0.5:", type(model_v05_saved).__name__)

print("\nSAVED PREDICTIONS")
print("Rows:", f"{len(scores_saved):,}")
print("FDA-linked events:", int(scores_saved["shortage_target"].sum()))

print("\nSAVED MODEL FILES VERIFIED")
print("No models retrained. No files modified.")

In [ ]:
# CELL 71 — SAVED MODEL PREDICTION REPRODUCIBILITY
# Read-only. No retraining or file changes.

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\jpiza\PharmaOpt")

panel_path = (
    ROOT / "outputs"
    / "recall_timing_corrected_v01"
    / "pharmaopt_package_year_recall_timing_v01.csv"
)

assert panel_path.is_file()

required = [
    "model_v04_saved",
    "model_v05_saved",
    "scores_saved"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}. Run Cell 70.")

# Extract features from the fitted model pipelines.
def model_features(model):
    transformer = model.named_steps["preprocessor"]
    columns = []

    for name, _, feature_names in transformer.transformers:
        if name == "remainder":
            continue
        columns.extend(list(feature_names))

    return list(dict.fromkeys(columns))

features04 = model_features(model_v04_saved)
features05 = model_features(model_v05_saved)

needed = list(dict.fromkeys(
    ["package_ndc", "prediction_year"] + features04 + features05
))

# Load only the 2025 columns needed for prediction.
panel = pd.read_csv(
    panel_path,
    usecols=needed,
    dtype={"package_ndc": "string"},
    low_memory=False
)

panel["package_ndc"] = panel["package_ndc"].str.strip()

panel_2025 = panel.loc[
    panel["prediction_year"] == 2025
].copy()

assert len(panel_2025) == 221_667
assert not panel_2025["package_ndc"].duplicated().any()

saved = scores_saved.copy()
saved["package_ndc"] = (
    saved["package_ndc"].astype("string").str.strip()
)

# Align rows by unique package-year keys.
aligned = saved[
    ["package_ndc", "prediction_year",
     "risk_score_v04", "risk_score_v05"]
].merge(
    panel_2025,
    on=["package_ndc", "prediction_year"],
    how="inner",
    validate="one_to_one"
)

assert len(aligned) == len(saved)

print("PHARMAOPT — PREDICTION REPRODUCIBILITY")
print("=" * 65)

results = []

for version, model, features in [
    ("v0.4", model_v04_saved, features04),
    ("v0.5", model_v05_saved, features05),
]:

    print(f"Recalculating {version} predictions...", flush=True)

    fresh = model.predict_proba(
        aligned[features]
    )[:, 1]

    stored = aligned[
        "risk_score_v04" if version == "v0.4"
        else "risk_score_v05"
    ].to_numpy(dtype=float)

    difference = np.abs(fresh - stored)

    results.append({
        "model": version,
        "evaluated_rows": len(fresh),
        "maximum_absolute_difference": float(difference.max()),
        "rows_differing_above_1e-10": int((difference > 1e-10).sum()),
    })

report = pd.DataFrame(results)
display(report)

assert (report["rows_differing_above_1e-10"] == 0).all()

print("\nPREDICTION REPRODUCIBILITY PASSED")
print("Saved models reproduce exported scores.")
print("Original datasets and model files unchanged.")

In [ ]:
# CELL 72 — PHARMAOPT DASHBOARD INVENTORY
# Read-only. No modifications.

from pathlib import Path

root = Path(r"C:\Users\jpiza\PharmaOpt")

print("PHARMAOPT — DASHBOARD INVENTORY")
print("=" * 60)

for path in sorted(root.rglob("*.py")):
    if ".venv" in path.parts:
        continue

    try:
        source = path.read_text(encoding="utf-8")
    except (UnicodeError, OSError):
        continue

    if (
        "streamlit" in source.lower()
        or path.name.lower() in ["app.py", "dashboard.py"]
    ):
        print("\nFILE:", path.relative_to(root))
        print("Contains Streamlit:", "import streamlit" in source)
        print("Lines:", len(source.splitlines()))

        for number, line in enumerate(
            source.splitlines(), start=1
        ):
            if any(
                term in line.lower()
                for term in [
                    "read_csv",
                    "risk_score",
                    "model_scores",
                    "shortage",
                    "st.title",
                    "st.set_page_config",
                ]
            ):
                print(f"  {number}: {line.strip()[:140]}")

print("\nDASHBOARD INVENTORY COMPLETE")